# Stage 3 — Training & Feature Selection: Temporal Matching, Primary Cohort Export, Ablation & Nested CV

**Module Role (`src/train`):** Executes **Studies 1–6** on the primary ADNI4 datasets:
- **Study 1:** Historical Prior ECog-12 Multimodal Baseline
- **Study 2:** Temporal Lookback Window Sensitivity Analysis
- **Study 3:** Fixed 30-Day Cohort Temporal Comparison → **saves the single primary cohort** (`data/processed/primary_multimodal_30day_cohort.csv`)
- **Study 4:** Leave-One-Item-Out ECog-12 Ablation
- **Study 5:** Repeated-Seed ECog-12 Ablation Stability
- **Study 6:** Nested Participant-Grouped Feature Selection

**Run Order:** Step 2 (run after `01_preprocessing.ipynb`).


## Setup: Load source datasets and configure output folders


In [47]:
# Load source processed datasets (Storyteller + ECog-12 PT) and configure output folders.
# The primary 30-day matched multimodal cohort (77 assessments / 71 participants) is built
# and saved in 03_train_temporal_and_ablation.ipynb as primary_multimodal_30day_cohort.csv.

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_DIR = None
DATA_DIR = None
for folder in [Path.cwd(), *Path.cwd().parents]:
    for candidate in (folder / "data" / "processed", folder / "processed_data"):
        if candidate.is_dir() and all(
            (candidate / filename).is_file()
            for filename in (
                "dataset1_storyteller.csv",
                "dataset2_ecog.csv",
            )
        ):
            PROJECT_DIR = folder
            DATA_DIR = candidate
            break
    if DATA_DIR is not None:
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find data/processed/ containing dataset1_storyteller.csv and dataset2_ecog.csv."
    )

RESULTS_ROOT = PROJECT_DIR / "results"
FIGURES_ROOT = PROJECT_DIR / "figures"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
FIGURES_ROOT.mkdir(parents=True, exist_ok=True)
PRIMARY_COHORT_CSV = DATA_DIR / "primary_multimodal_30day_cohort.csv"

dataset1_storyteller = pd.read_csv(DATA_DIR / "dataset1_storyteller.csv", low_memory=False)
dataset2_ecog = pd.read_csv(DATA_DIR / "dataset2_ecog.csv", low_memory=False)

# Normalize identifiers, targets, and dates without changing cohort membership.
for df in (dataset1_storyteller, dataset2_ecog):
    if "PTID" not in df or "TARGET" not in df:
        raise KeyError("Each dataset must contain PTID and TARGET columns.")
    df["PTID"] = df["PTID"].astype("string").str.strip()
    df["TARGET"] = pd.to_numeric(df["TARGET"], errors="coerce")
    if df[["PTID", "TARGET"]].isna().any().any():
        raise ValueError("A dataset contains missing PTID or TARGET values.")
    if not df["TARGET"].isin([0, 1]).all():
        raise ValueError("TARGET must contain only 0 and 1.")
    df["TARGET"] = df["TARGET"].astype(int)
    for col in df.columns:
        if "DATE" in str(col).upper():
            df[col] = pd.to_datetime(df[col], errors="coerce")

# Apply the feature exclusions used in the predictor audit (51 Storyteller, 12 ECOG).
STORY_EXCLUDED_TERMS = ("PLAYBACK_COUNT", "GMATCH", "LANGUAGE")
story_feature_cols = [
    col for col in dataset1_storyteller.columns
    if str(col).upper().startswith("STORY")
    and not any(term in str(col).upper() for term in STORY_EXCLUDED_TERMS)
]
for col in story_feature_cols:
    dataset1_storyteller[col] = pd.to_numeric(dataset1_storyteller[col], errors="coerce")
story_feature_cols = [
    col for col in story_feature_cols
    if dataset1_storyteller[col].notna().any()
]

ecog_feature_cols = [f"ECOGPT{i}" for i in range(1, 13)]
missing_ecog = [col for col in ecog_feature_cols if col not in dataset2_ecog.columns]
if missing_ecog:
    raise KeyError(f"Dataset 2 is missing predictor columns: {missing_ecog}")
for col in ecog_feature_cols:
    dataset2_ecog[col] = pd.to_numeric(dataset2_ecog[col], errors="coerce")

print("Loaded source datasets from:", DATA_DIR)
for name, df in (
    ("Storyteller Source Cohort", dataset1_storyteller),
    ("ECog-12 PT Source Cohort", dataset2_ecog),
):
    print(
        f"{name}: {len(df)} rows, {df['PTID'].nunique()} participants, "
        f"targets={df['TARGET'].value_counts().sort_index().to_dict()}"
    )
print(
    f"Features: {len(story_feature_cols)} Storyteller, "
    f"{len(ecog_feature_cols)} ECog-12 PT, "
    f"{len(story_feature_cols) + len(ecog_feature_cols)} full fusion"
)


Loaded source datasets from: c:\Users\amnar\Documents\ADNI4_Storyteller_ECOG_Classification\data\processed
Storyteller Source Cohort: 1430 rows, 490 participants, targets={0: 1073, 1: 357}
ECog-12 PT Source Cohort: 501 rows, 397 participants, targets={0: 273, 1: 228}
Features: 51 Storyteller, 12 ECog-12 PT, 63 full fusion


## Study 1 — Historical Prior ECog-12 Multimodal Baseline

Evaluates Storyteller-only, historical prior ECog-12 PT, and combined multimodal baselines using all prior ECog-12 records dated on or before each Storyteller assessment.


In [48]:
# Historical Prior ECOG Multimodal Classification HISTORICAL PRIOR-ECOG MULTIMODAL CLASSIFICATION
#
# Goal:
# Use ECOG records collected on or before each corresponding
# Storyteller assessment. This has no maximum look-back period:
# an ECOG record may be many months or years before Storyteller.
# Treat this as historical prior-record matching, not a
# contemporaneous or fixed-window analysis. Study 3: Fixed 30-Day Cohort Comparison is the
# fixed-cohort temporal-window comparison.
#
# Models:
#   1. Storyteller-only
#   2. ECOG-only
#   3. Storyteller + ECOG
#
# Evaluation:
#   5-fold StratifiedGroupKFold
#   Group = PTID
#
# Primary evaluation: assessment-level
# Secondary evaluation: participant-level, for participants
# whose diagnosis labels are consistent across assessments.
# ============================================================


# ------------------------------------------------------------
# Study 1: Historical Prior ECOG Baseline model imports and settings
# ------------------------------------------------------------

import numpy as np
import pandas as pd
import warnings

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
N_SPLITS = 5

# ------------------------------------------------------------
# 1. CONFIGURATION: DATE COLUMN DETECTION
# ------------------------------------------------------------

# Preferred column names based on the prepared ADNI data.
# If your columns have different names, update these candidates.

STORY_DATE_CANDIDATES = [
    "EXAMDATE_STORY",
    "EXAMDATE",
    "STORY_DATE",
    "COLDATE_STORY"
]

ECOG_DATE_CANDIDATES = [
    "COLDATE",
    "EXAMDATE_ECOG",
    "EXAMDATE",
    "ECOG_DATE"
]

def find_date_column(df, candidates, dataset_name):

    for col in candidates:
        if col in df.columns:
            print(f"{dataset_name} date column:", col)
            return col

    raise ValueError(
        f"Could not find the date column for {dataset_name}.\n"
        f"Available columns:\n{df.columns.tolist()}\n"
        f"Update the date candidates in Section 1."
    )


story_date_col = find_date_column(
    dataset1_storyteller,
    STORY_DATE_CANDIDATES,
    "Storyteller"
)

ecog_date_col = find_date_column(
    dataset2_ecog,
    ECOG_DATE_CANDIDATES,
    "ECOG"
)

# ------------------------------------------------------------
# 2. PREPARE STORYTELLER DATA
# ------------------------------------------------------------

story17 = dataset1_storyteller.copy()
ecog17 = dataset2_ecog.copy()

# Normalize participant identifiers
for df in [story17, ecog17]:

    df["PTID"] = (
        df["PTID"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

# Convert diagnosis labels
story17["TARGET"] = pd.to_numeric(
    story17["TARGET"],
    errors="coerce"
)

# Retain CN and MCI only
story17 = story17[
    story17["TARGET"].isin([0, 1]) &
    story17["PTID"].notna() &
    (story17["PTID"] != "")
].copy()

story17["TARGET"] = story17["TARGET"].astype(int)

# Parse dates
story17[story_date_col] = pd.to_datetime(
    story17[story_date_col],
    errors="coerce"
)

ecog17[ecog_date_col] = pd.to_datetime(
    ecog17[ecog_date_col],
    errors="coerce"
)

# Remove records without usable dates
story17 = story17[
    story17[story_date_col].notna()
].copy()

ecog17 = ecog17[
    ecog17[ecog_date_col].notna() &
    ecog17["PTID"].notna() &
    (ecog17["PTID"] != "")
].copy()

# ------------------------------------------------------------
# 3. PREPARE FEATURES
# ------------------------------------------------------------

for col in story_feature_cols:

    story17[col] = pd.to_numeric(
        story17[col],
        errors="coerce"
    )

for col in ecog_feature_cols:

    ecog17[col] = pd.to_numeric(
        ecog17[col],
        errors="coerce"
    )

# ECOG code 9 represents missing
ecog17[ecog_feature_cols] = (
    ecog17[ecog_feature_cols].replace(9, np.nan)
)

# Confirm feature columns
assert len(story_feature_cols) == 51
assert len(ecog_feature_cols) == 12

# ------------------------------------------------------------
# 4. TEMPORAL ECOG MATCHING
# ------------------------------------------------------------

# For every Storyteller assessment:
#   - Find ECOG records from the same participant.
#   - Keep only records collected on or before Storyteller date.
#   - Aggregate their ECOG responses using the median.
#
# This is an unlimited-look-back historical aggregation.
# It does NOT use ECOG records collected after the corresponding
# Storyteller assessment, but the most recent prior record may be
# distant in time. Interpret results as historical matching.

ecog_by_participant = {
    ptid: group.sort_values(ecog_date_col).copy()
    for ptid, group in ecog17.groupby("PTID")
}

temporal_records = []

for story_index, story_row in story17.iterrows():

    ptid = story_row["PTID"]
    story_date = story_row[story_date_col]

    participant_ecog = ecog_by_participant.get(ptid)

    record = {
        "STORY_INDEX": story_index,
        "PTID": ptid,
        "TARGET": int(story_row["TARGET"]),
        "STORY_DATE": story_date
    }

    if participant_ecog is None:

        record["ECOG_RECORD_COUNT"] = 0
        record["LATEST_ECOG_DATE"] = pd.NaT
        record["DAYS_SINCE_LATEST_ECOG"] = np.nan

        for col in ecog_feature_cols:
            record[col] = np.nan

    else:

        # Strict temporal restriction
        available_ecog = participant_ecog[
            participant_ecog[ecog_date_col] <= story_date
        ]

        if len(available_ecog) == 0:

            record["ECOG_RECORD_COUNT"] = 0
            record["LATEST_ECOG_DATE"] = pd.NaT
            record["DAYS_SINCE_LATEST_ECOG"] = np.nan

            for col in ecog_feature_cols:
                record[col] = np.nan

        else:

            # Aggregate only ECOG records available by this date
            ecog_medians = available_ecog[
                ecog_feature_cols
            ].median()

            record["ECOG_RECORD_COUNT"] = len(
                available_ecog
            )

            latest_date = available_ecog[
                ecog_date_col
            ].max()

            record["LATEST_ECOG_DATE"] = latest_date

            record["DAYS_SINCE_LATEST_ECOG"] = (
                story_date - latest_date
            ).days

            for col in ecog_feature_cols:
                record[col] = ecog_medians[col]

    temporal_records.append(record)


temporal_ecog = pd.DataFrame(temporal_records)

# Restore Storyteller feature values
story_features = story17[
    ["PTID", "TARGET", story_date_col] +
    list(story_feature_cols)
].copy()

story_features = story_features.rename(
    columns={story_date_col: "STORY_DATE"}
)

# Join temporal ECOG features to corresponding Storyteller rows
temporal_df = story_features.merge(
    temporal_ecog.drop(columns=["TARGET", "PTID", "STORY_DATE"]),
    left_index=True,
    right_on="STORY_INDEX",
    how="inner"
)

temporal_df = temporal_df.reset_index(drop=True)

# ------------------------------------------------------------
# 5. TEMPORAL MATCHING AUDIT
# ------------------------------------------------------------

total_assessments = len(temporal_df)

has_ecog = (
    temporal_df["ECOG_RECORD_COUNT"] > 0
)

matched_df = temporal_df[
    has_ecog
].copy()

unmatched_df = temporal_df[
    ~has_ecog
].copy()

print("\n" + "=" * 65)
print("STUDY 1: HISTORICAL PRIOR ECOG BASELINE HISTORICAL PRIOR-ECOG MATCHING AUDIT")
print("=" * 65)

print("Total Storyteller assessments:", total_assessments)

print(
    "Assessments with prior/on-date ECOG:",
    len(matched_df)
)

print(
    "Assessments without prior/on-date ECOG:",
    len(unmatched_df)
)

print(
    "Participants with temporally available ECOG:",
    matched_df["PTID"].nunique()
)

print(
    "Participants with no matched Storyteller assessment:",
    unmatched_df["PTID"].nunique()
)

matched_ptids17 = set(matched_df["PTID"].astype(str))
unmatched_ptids17 = set(unmatched_df["PTID"].astype(str))
print(
    "Participants appearing in both assessment groups:",
    len(matched_ptids17 & unmatched_ptids17)
)
print(
    "Participants represented in either group:",
    len(matched_ptids17 | unmatched_ptids17)
)
print(
    "Note: matched and unmatched participant counts can overlap; "
    "assessment rows, not participants, form a partition."
)

print(
    "Percentage of assessments with ECOG:",
    round(100 * len(matched_df) / total_assessments, 2),
    "%"
)

print("\nECOG records available per matched assessment:")
display(
    matched_df["ECOG_RECORD_COUNT"].describe().to_frame()
)

print("\nDays between latest available ECOG and Storyteller:")
display(
    matched_df["DAYS_SINCE_LATEST_ECOG"].describe().to_frame()
)

# ------------------------------------------------------------
# 6. BUILD MATCHED FEATURE MATRICES
# ------------------------------------------------------------

# Fair comparison:
# All three models use exactly the same assessments.

X17_story = matched_df[
    story_feature_cols
].copy()

X17_ecog = matched_df[
    ecog_feature_cols
].copy()

X17_fusion = matched_df[
    list(story_feature_cols) +
    list(ecog_feature_cols)
].copy()

y17 = matched_df["TARGET"].to_numpy()
groups17 = matched_df["PTID"].to_numpy()

print("\n" + "=" * 65)
print("MATCHED EXPERIMENT DATA SUMMARY")
print("=" * 65)

print("Matched assessments:", len(matched_df))
print("Matched participants:", matched_df["PTID"].nunique())

print("\nDiagnosis distribution:")
print(
    matched_df["TARGET"]
    .value_counts()
    .sort_index()
    .rename(index={0: "CN", 1: "MCI"})
)

print("\nFeature dimensions:")
print("Storyteller:", X17_story.shape)
print("ECOG:", X17_ecog.shape)
print("Fusion:", X17_fusion.shape)

# ------------------------------------------------------------
# 7. MODEL FUNCTION
# ------------------------------------------------------------

def create_exp17_model():

    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
                keep_empty_features=True
            )
        ),
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
                random_state=RANDOM_STATE
            )
        )
    ])


feature_sets17 = {
    "Storyteller": X17_story,
    "ECOG": X17_ecog,
    "Multimodal": X17_fusion
}

# ------------------------------------------------------------
# 8. PARTICIPANT-GROUPED CROSS-VALIDATION
# ------------------------------------------------------------

cv17 = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

oof17 = {
    name: np.full(len(matched_df), np.nan)
    for name in feature_sets17
}

print("\n" + "=" * 65)
print("STUDY 1: HISTORICAL PRIOR ECOG BASELINE MODEL TRAINING")
print("=" * 65)

for fold, (train_idx, valid_idx) in enumerate(
    cv17.split(X17_fusion, y17, groups17),
    start=1
):

    print(f"\nFold {fold}/{N_SPLITS}")

    # Leakage check
    train_participants = set(groups17[train_idx])
    valid_participants = set(groups17[valid_idx])

    assert train_participants.isdisjoint(
        valid_participants
    )

    print("Training participants:", len(train_participants))
    print("Validation participants:", len(valid_participants))

    for model_name, X_current in feature_sets17.items():

        model = create_exp17_model()

        model.fit(
            X_current.iloc[train_idx],
            y17[train_idx]
        )

        probabilities = model.predict_proba(
            X_current.iloc[valid_idx]
        )[:, 1]

        oof17[model_name][valid_idx] = probabilities

print("\nCross-validation completed.")

# ------------------------------------------------------------
# 9. EVALUATION FUNCTION
# ------------------------------------------------------------

def evaluate_exp17(y_true, probabilities):

    predictions = (
        probabilities >= 0.50
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "Accuracy": accuracy_score(
            y_true, predictions
        ),

        "Balanced Accuracy": balanced_accuracy_score(
            y_true, predictions
        ),

        "MCI Precision": precision_score(
            y_true, predictions,
            zero_division=0
        ),

        "MCI Recall": recall_score(
            y_true, predictions,
            zero_division=0
        ),

        "MCI F1": f1_score(
            y_true, predictions,
            zero_division=0
        ),

        "Macro F1": f1_score(
            y_true, predictions,
            average="macro",
            zero_division=0
        ),

        "ROC-AUC": roc_auc_score(
            y_true, probabilities
        ),

        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }


# ------------------------------------------------------------
# 10. ASSESSMENT-LEVEL RESULTS
# ------------------------------------------------------------

results17_assessment = []

for model_name, probabilities in oof17.items():

    if np.isnan(probabilities).any():
        raise ValueError(
            f"Missing OOF predictions for {model_name}"
        )

    metrics = evaluate_exp17(
        y17,
        probabilities
    )

    metrics["Model"] = model_name
    metrics["Evaluation"] = "Assessment-level"

    results17_assessment.append(metrics)


results17_assessment_df = pd.DataFrame(
    results17_assessment
)

print("\n" + "=" * 65)
print("STUDY 1: HISTORICAL PRIOR ECOG BASELINE ASSESSMENT-LEVEL RESULTS")
print("=" * 65)

display(
    results17_assessment_df.round(4)
)

# ------------------------------------------------------------
# 11. PARTICIPANT-LEVEL RESULTS
# ------------------------------------------------------------

participant17 = pd.DataFrame({
    "PTID": groups17,
    "TARGET": y17
})

for model_name, probabilities in oof17.items():
    participant17[model_name] = probabilities

# Check diagnosis consistency before participant aggregation
participant_label_counts = (
    participant17.groupby("PTID")["TARGET"].nunique()
)

consistent_ids = participant_label_counts[
    participant_label_counts == 1
].index

inconsistent_ids = participant_label_counts[
    participant_label_counts > 1
].index

participant17_consistent = participant17[
    participant17["PTID"].isin(consistent_ids)
].copy()

participant17_results = (
    participant17_consistent
    .groupby("PTID", as_index=False)
    .agg({
        "TARGET": "first",
        **{
            name: "mean"
            for name in oof17.keys()
        }
    })
)

print("\nParticipants with consistent labels:",
      len(participant17_results))

print("Participants with changing labels excluded:",
      len(inconsistent_ids))

results17_participant = []

for model_name in oof17:

    metrics = evaluate_exp17(
        participant17_results["TARGET"].to_numpy(),
        participant17_results[model_name].to_numpy()
    )

    metrics["Model"] = model_name
    metrics["Evaluation"] = "Participant-level"

    results17_participant.append(metrics)


results17_participant_df = pd.DataFrame(
    results17_participant
)

print("\n" + "=" * 65)
print("STUDY 1: HISTORICAL PRIOR ECOG BASELINE PARTICIPANT-LEVEL RESULTS")
print("=" * 65)

display(
    results17_participant_df.round(4)
)




# ============================================================


Storyteller date column: EXAMDATE_STORY
ECOG date column: COLDATE

STUDY 1: HISTORICAL PRIOR ECOG BASELINE HISTORICAL PRIOR-ECOG MATCHING AUDIT
Total Storyteller assessments: 1430
Assessments with prior/on-date ECOG: 515
Assessments without prior/on-date ECOG: 915
Participants with temporally available ECOG: 185
Participants with no matched Storyteller assessment: 336
Participants appearing in both assessment groups: 31
Participants represented in either group: 490
Note: matched and unmatched participant counts can overlap; assessment rows, not participants, form a partition.
Percentage of assessments with ECOG: 36.01 %

ECOG records available per matched assessment:


,ECOG_RECORD_COUNT
count,515.000000
mean,1.066019
std,0.271024
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,3.000000



Days between latest available ECOG and Storyteller:


,DAYS_SINCE_LATEST_ECOG
count,515.000000
mean,266.955340
std,211.274841
min,1.000000
25%,59.500000
50%,222.000000
75%,401.000000
max,988.000000



MATCHED EXPERIMENT DATA SUMMARY
Matched assessments: 515
Matched participants: 185

Diagnosis distribution:
TARGET
CN     382
MCI    133
Name: count, dtype: int64

Feature dimensions:
Storyteller: (515, 51)
ECOG: (515, 12)
Fusion: (515, 63)

STUDY 1: HISTORICAL PRIOR ECOG BASELINE MODEL TRAINING

Fold 1/5
Training participants: 149
Validation participants: 36

Fold 2/5
Training participants: 148
Validation participants: 37

Fold 3/5
Training participants: 148
Validation participants: 37

Fold 4/5
Training participants: 148
Validation participants: 37

Fold 5/5
Training participants: 147
Validation participants: 38

Cross-validation completed.

STUDY 1: HISTORICAL PRIOR ECOG BASELINE ASSESSMENT-LEVEL RESULTS


,Accuracy,Balanced Accuracy,MCI Precision,MCI Recall,MCI F1,Macro F1,ROC-AUC,TN,FP,FN,TP,Model,Evaluation
0,0.6757,0.6417,0.4086,0.5714,0.4765,0.6208,0.6974,272,110,57,76,Storyteller,Assessment-level
1,0.7223,0.6756,0.4695,0.5789,0.5185,0.6617,0.7187,295,87,56,77,ECOG,Assessment-level
2,0.7320,0.6944,0.4852,0.6165,0.5430,0.6767,0.7431,295,87,51,82,Multimodal,Assessment-level



Participants with consistent labels: 182
Participants with changing labels excluded: 3

STUDY 1: HISTORICAL PRIOR ECOG BASELINE PARTICIPANT-LEVEL RESULTS


,Accuracy,Balanced Accuracy,MCI Precision,MCI Recall,MCI F1,Macro F1,ROC-AUC,TN,FP,FN,TP,Model,Evaluation
0,0.7198,0.7083,0.5352,0.6786,0.5984,0.6916,0.7440,93,33,18,38,Storyteller,Participant-level
1,0.6868,0.6448,0.4918,0.5357,0.5128,0.6410,0.6806,95,31,26,30,ECOG,Participant-level
2,0.7088,0.6806,0.5231,0.6071,0.5620,0.6719,0.7252,95,31,22,34,Multimodal,Participant-level


In [49]:
RESULTS_DIR_17=RESULTS_ROOT / "historical_prior_ecog_classification"
RESULTS_DIR_17.mkdir(parents=True,exist_ok=True)
results17_assessment_df.to_csv(RESULTS_DIR_17 / "assessment_level_metrics.csv",index=False)
results17_participant_df.to_csv(RESULTS_DIR_17 / "participant_level_metrics.csv",index=False)


## Study 2 — Temporal Lookback Window Sensitivity Analysis

Tests how restricting the maximum Storyteller–ECog-12 lookback window (30, 90, 180, 365 days vs. unrestricted historical prior) affects cohort size and participant-level discrimination.


In [50]:
# Temporal Window Sensitivity Analysis ECOG TEMPORAL-WINDOW SENSITIVITY ANALYSIS
#
# Cohort note: this experiment rebuilds the eligible cohort separately
# for each window. Window metrics therefore combine changes in ECOG
# look-back and changes in which assessments/participants are included.
# Use Study 3: Fixed 30-Day Cohort Comparison for the fixed-cohort, same-assessment comparison.
#
# Windows:
#   1. 30 days
#   2. 90 days
#   3. 180 days
#   4. 365 days
#   5. Any historical ECOG measurement before/on Storyteller
#
# Models:
#   A. Storyteller-only
#   B. ECOG-only
#   C. Storyteller + ECOG
#
# Evaluation:
#   - 5-fold StratifiedGroupKFold
#   - Grouping by participant (PTID)
#   - Assessment-level and participant-level metrics
#   - Same eligible assessments for all three models
# ============================================================

import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

WINDOWS = {
    "30_days": 30,
    "90_days": 90,
    "180_days": 180,
    "365_days": 365,
    "historical": None
}

N_SPLITS = 5
RANDOM_STATE = 42
THRESHOLD = 0.50

# ------------------------------------------------------------
# 2. PREPARE STORYTELLER DATA
# ------------------------------------------------------------

story18 = dataset1_storyteller.copy().reset_index(drop=True)
ecog18 = dataset2_ecog.copy().reset_index(drop=True)

required_story_cols = (
    ["PTID", "TARGET", "EXAMDATE_STORY"] + list(story_feature_cols)
)
required_ecog_cols = (
    ["PTID", "COLDATE"] + list(ecog_feature_cols)
)

missing_story = [
    c for c in required_story_cols if c not in story18.columns
]
missing_ecog = [
    c for c in required_ecog_cols if c not in ecog18.columns
]

if missing_story:
    raise ValueError(f"Missing Storyteller columns: {missing_story}")

if missing_ecog:
    raise ValueError(f"Missing ECOG columns: {missing_ecog}")

# Normalize participant identifiers
story18["PTID"] = (
    story18["PTID"].astype("string").str.strip().str.upper()
)

ecog18["PTID"] = (
    ecog18["PTID"].astype("string").str.strip().str.upper()
)

# Keep only CN and MCI Storyteller assessments
story18["TARGET"] = pd.to_numeric(
    story18["TARGET"], errors="coerce"
)

story18 = story18[
    story18["TARGET"].isin([0, 1]) &
    story18["PTID"].notna()
].copy()

story18["TARGET"] = story18["TARGET"].astype(int)

# Parse dates
story18["EXAMDATE_STORY"] = pd.to_datetime(
    story18["EXAMDATE_STORY"], errors="coerce"
)

ecog18["COLDATE"] = pd.to_datetime(
    ecog18["COLDATE"], errors="coerce"
)

# Remove records without usable participant/date information
story18 = story18.dropna(
    subset=["PTID", "EXAMDATE_STORY"]
).reset_index(drop=True)

ecog18 = ecog18.dropna(
    subset=["PTID", "COLDATE"]
).copy()

# Convert Storyteller predictors to numeric
for col in story_feature_cols:
    story18[col] = pd.to_numeric(
        story18[col], errors="coerce"
    )

# Convert ECOG predictors to numeric
for col in ecog_feature_cols:
    ecog18[col] = pd.to_numeric(
        ecog18[col], errors="coerce"
    )

# ECOG value 9 represents missing
ecog18[ecog_feature_cols] = (
    ecog18[ecog_feature_cols].replace(9, np.nan)
)

# Keep ECOG records with at least one observed questionnaire item
ecog18 = ecog18[
    ecog18[ecog_feature_cols].notna().any(axis=1)
].copy()

# Sort ECOG records by participant and date
ecog18 = ecog18.sort_values(
    ["PTID", "COLDATE"]
).reset_index(drop=True)

# Store ECOG records by participant for faster matching
ecog_by_ptid = {
    ptid: group.copy()
    for ptid, group in ecog18.groupby("PTID")
}

print("=" * 65)
print("Temporal Window Sensitivity Analysis DATA PREPARATION")
print(
    "Interpretation: each time window may have a different eligible cohort; "
    "use Study 3: Fixed 30-Day Cohort Comparison for fixed-cohort window comparisons."
)
print("=" * 65)

print("Storyteller assessments:", len(story18))
print("Storyteller participants:", story18["PTID"].nunique())
print("ECOG records:", len(ecog18))
print("ECOG participants:", ecog18["PTID"].nunique())
print("Storyteller features:", len(story_feature_cols))
print("ECOG features:", len(ecog_feature_cols))

print("\nStoryteller diagnosis distribution:")
print(story18["TARGET"].value_counts().sort_index())

# ------------------------------------------------------------
# 3. FUNCTION TO BUILD A TEMPORAL-WINDOW DATASET
# ------------------------------------------------------------

def build_temporal_window_dataset(window_name, window_days):

    matched_rows = []
    match_counts = []
    latest_gaps = []

    for row_number, story_row in story18.iterrows():

        ptid = story_row["PTID"]
        story_date = story_row["EXAMDATE_STORY"]

        # Find ECOG records for this participant
        participant_ecog = ecog_by_ptid.get(ptid)

        if participant_ecog is None:
            continue

        # ECOG must be collected on or before Storyteller
        eligible_ecog = participant_ecog[
            participant_ecog["COLDATE"] <= story_date
        ].copy()

        # Calculate number of days before Storyteller
        eligible_ecog["GAP_DAYS"] = (
            story_date - eligible_ecog["COLDATE"]
        ).dt.days

        # Apply temporal window
        if window_days is not None:
            eligible_ecog = eligible_ecog[
                eligible_ecog["GAP_DAYS"] <= window_days
            ]

        if eligible_ecog.empty:
            continue

        # Aggregate ECOG items using the median of eligible records
        ecog_medians = eligible_ecog[
            ecog_feature_cols
        ].median(axis=0)

        # Require at least one usable ECOG feature
        if ecog_medians.notna().sum() == 0:
            continue

        # Preserve Storyteller assessment and its label
        new_row = story_row.to_dict()

        for col in ecog_feature_cols:
            new_row[col] = ecog_medians[col]

        new_row["N_ECOG_RECORDS"] = len(eligible_ecog)
        new_row["LATEST_ECOG_GAP_DAYS"] = (
            eligible_ecog["GAP_DAYS"].min()
        )

        matched_rows.append(new_row)
        match_counts.append(len(eligible_ecog))
        latest_gaps.append(
            eligible_ecog["GAP_DAYS"].min()
        )

    matched_df = pd.DataFrame(matched_rows)

    if matched_df.empty:
        return matched_df, {}

    audit = {
        "window": window_name,
        "window_days": (
            "Any historical" if window_days is None
            else window_days
        ),
        "assessments": len(matched_df),
        "participants": matched_df["PTID"].nunique(),
        "CN_assessments": int(
            (matched_df["TARGET"] == 0).sum()
        ),
        "MCI_assessments": int(
            (matched_df["TARGET"] == 1).sum()
        ),
        "mean_ECOG_records": np.mean(match_counts),
        "median_latest_gap_days": np.median(latest_gaps),
        "mean_latest_gap_days": np.mean(latest_gaps),
        "max_latest_gap_days": np.max(latest_gaps)
    }

    return matched_df, audit


# ------------------------------------------------------------
# 4. METRIC CALCULATION
# ------------------------------------------------------------

def calculate_window_metrics(y_true, probabilities, threshold=0.50):

    predictions = (
        np.asarray(probabilities) >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "Accuracy": accuracy_score(y_true, predictions),
        "Balanced_Accuracy": balanced_accuracy_score(
            y_true, predictions
        ),
        "MCI_Precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "MCI_Recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "MCI_F1": f1_score(
            y_true, predictions, zero_division=0
        ),
        "Macro_F1": f1_score(
            y_true, predictions, average="macro",
            zero_division=0
        ),
        "ROC_AUC": (
            roc_auc_score(y_true, probabilities)
            if len(np.unique(y_true)) == 2
            else np.nan
        ),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }


# ------------------------------------------------------------
# 5. MODEL PIPELINE
# ------------------------------------------------------------

def create_experiment18_model():

    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
                keep_empty_features=True
            )
        ),
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=3000,
                random_state=RANDOM_STATE
            )
        )
    ])


# ------------------------------------------------------------
# 6. RUN GROUPED CROSS-VALIDATION FOR EACH WINDOW
# ------------------------------------------------------------

all_assessment_results = []
all_participant_results = []
all_audit_results = []
all_predictions = []

for window_name, window_days in WINDOWS.items():

    print("\n" + "=" * 65)
    print(f"RUNNING WINDOW: {window_name}")
    print("=" * 65)

    matched_df, audit = build_temporal_window_dataset(
        window_name, window_days
    )

    if matched_df.empty:
        print("No eligible assessments. Skipping.")
        continue

    # Check that both classes are present
    if matched_df["TARGET"].nunique() < 2:
        print("Only one diagnosis class found. Skipping.")
        continue

    X_story = matched_df[story_feature_cols].copy()
    X_ecog = matched_df[ecog_feature_cols].copy()
    X_fusion = matched_df[
        story_feature_cols + ecog_feature_cols
    ].copy()

    y = matched_df["TARGET"].to_numpy()
    groups = matched_df["PTID"].to_numpy()

    # Count participants represented in each class
    participant_labels = (
        matched_df[["PTID", "TARGET"]]
        .drop_duplicates()
    )

    participants_per_class = (
        participant_labels.groupby("TARGET")["PTID"]
        .nunique()
    )

    min_class_participants = int(
        participants_per_class.min()
    )

    # Five folds require at least five participant groups
    # in each class for a meaningful stratified evaluation.
    if min_class_participants < N_SPLITS:
        print(
            "Insufficient participant groups in one class "
            f"for {N_SPLITS}-fold CV: {participants_per_class.to_dict()}"
        )
        print("Skipping this window rather than producing unstable folds.")
        continue

    print("\nWindow cohort:")
    print("Assessments:", len(matched_df))
    print("Participants:", matched_df["PTID"].nunique())
    print("CN assessments:", int((y == 0).sum()))
    print("MCI assessments:", int((y == 1).sum()))
    print("Participant counts by class:",
          participants_per_class.to_dict())

    print("\nTemporal audit:")
    for key, value in audit.items():
        print(f"{key}: {value}")

    # Save audit
    all_audit_results.append(audit)

    # All models use the same assessment rows and same folds
    model_inputs = {
        "Storyteller": X_story,
        "ECOG": X_ecog,
        "Multimodal": X_fusion
    }

    splitter = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    # OOF probability storage
    oof_probabilities = {
        model_name: np.full(len(y), np.nan)
        for model_name in model_inputs
    }

    fold_number = 0

    for train_idx, test_idx in splitter.split(
        X_fusion, y, groups
    ):

        fold_number += 1

        train_groups = set(groups[train_idx])
        test_groups = set(groups[test_idx])

        # Leakage check
        assert train_groups.isdisjoint(test_groups)

        print(
            f"Fold {fold_number}: "
            f"train participants={len(train_groups)}, "
            f"test participants={len(test_groups)}"
        )

        for model_name, X in model_inputs.items():

            model = create_experiment18_model()

            model.fit(
                X.iloc[train_idx],
                y[train_idx]
            )

            probabilities = model.predict_proba(
                X.iloc[test_idx]
            )[:, 1]

            oof_probabilities[model_name][test_idx] = (
                probabilities
            )

    # --------------------------------------------------------
    # 7. ASSESSMENT-LEVEL RESULTS
    # --------------------------------------------------------

    for model_name, probabilities in oof_probabilities.items():

        if np.isnan(probabilities).any():
            raise ValueError(
                f"Missing OOF predictions for {model_name}"
            )

        metrics = calculate_window_metrics(
            y, probabilities, THRESHOLD
        )

        result = {
            "Window": window_name,
            "Model": model_name,
            "Assessments": len(y),
            "Participants": len(np.unique(groups)),
            **metrics
        }

        all_assessment_results.append(result)

        print(
            f"\n{model_name} assessment-level results:"
        )

        for metric, value in metrics.items():
            print(f"{metric}: {value}")

        # Save individual assessment predictions
        prediction_df = matched_df[
            ["PTID", "EXAMDATE_STORY", "TARGET"]
        ].copy()

        prediction_df["Window"] = window_name
        prediction_df["Model"] = model_name
        prediction_df["Probability_MCI"] = probabilities
        prediction_df["Predicted_TARGET"] = (
            probabilities >= THRESHOLD
        ).astype(int)

        all_predictions.append(prediction_df)

    # --------------------------------------------------------
    # 8. PARTICIPANT-LEVEL RESULTS
    # --------------------------------------------------------

    # Average assessment probabilities for each participant.
    # Exclude participants whose diagnosis changes across
    # their included Storyteller assessments.
    participant_data = matched_df[
        ["PTID", "TARGET"]
    ].copy()

    consistent_ids = (
        participant_data.groupby("PTID")["TARGET"]
        .nunique()
    )

    consistent_ids = consistent_ids[
        consistent_ids == 1
    ].index

    participant_base = matched_df[
        matched_df["PTID"].isin(consistent_ids)
    ][["PTID", "TARGET"]].drop_duplicates()

    for model_name, probabilities in oof_probabilities.items():

        temp = matched_df[
            ["PTID", "TARGET"]
        ].copy()

        temp["Probability_MCI"] = probabilities

        participant_predictions = (
            temp[temp["PTID"].isin(consistent_ids)]
            .groupby("PTID")
            .agg(
                TARGET=("TARGET", "first"),
                Probability_MCI=("Probability_MCI", "mean")
            )
            .reset_index()
        )

        participant_metrics = calculate_window_metrics(
            participant_predictions["TARGET"],
            participant_predictions["Probability_MCI"],
            THRESHOLD
        )

        participant_result = {
            "Window": window_name,
            "Model": model_name,
            "Participants": len(participant_predictions),
            "Excluded_inconsistent_label_participants": (
                matched_df["PTID"].nunique()
                - len(participant_predictions)
            ),
            **participant_metrics
        }

        all_participant_results.append(participant_result)

        print(
            f"\n{model_name} participant-level results:"
        )

        for metric, value in participant_metrics.items():
            print(f"{metric}: {value}")


# ------------------------------------------------------------
# 9. CREATE RESULT DATAFRAMES
# ------------------------------------------------------------

experiment18_assessment_results = pd.DataFrame(
    all_assessment_results
)

experiment18_participant_results = pd.DataFrame(
    all_participant_results
)

experiment18_temporal_audit = pd.DataFrame(
    all_audit_results
)

experiment18_predictions = (
    pd.concat(all_predictions, ignore_index=True)
    if all_predictions else pd.DataFrame()
)

# ------------------------------------------------------------
# 10. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("Temporal Window Sensitivity Analysis TEMPORAL WINDOW AUDIT")
print("=" * 75)

display(experiment18_temporal_audit.round(3))

print("\n" + "=" * 75)
print("Temporal Window Sensitivity Analysis ASSESSMENT-LEVEL RESULTS")
print("=" * 75)

display(
    experiment18_assessment_results.sort_values(
        ["Window", "Model"]
    ).round(4)
)

print("\n" + "=" * 75)
print("Temporal Window Sensitivity Analysis PARTICIPANT-LEVEL RESULTS")
print("=" * 75)

display(
    experiment18_participant_results.sort_values(
        ["Window", "Model"]
    ).round(4)
)

# ------------------------------------------------------------
# 11. SAVE RESULTS
# ------------------------------------------------------------

from pathlib import Path

output_dir = RESULTS_ROOT / "temporal_window_sensitivity"
output_dir.mkdir(exist_ok=True)

experiment18_assessment_results.to_csv(
    output_dir / "temporal_window_sensitivity_assessment_results.csv",
    index=False
)

experiment18_participant_results.to_csv(
    output_dir / "temporal_window_sensitivity_participant_results.csv",
    index=False
)

experiment18_temporal_audit.to_csv(
    output_dir / "temporal_window_sensitivity_temporal_audit.csv",
    index=False
)

experiment18_predictions.to_csv(
    output_dir / "temporal_window_sensitivity_oof_predictions.csv",
    index=False
)

print("\nStudy 2: Temporal Window Sensitivity completed.")
print("Results saved to:", output_dir.resolve())



# ============================================================


Temporal Window Sensitivity Analysis DATA PREPARATION
Interpretation: each time window may have a different eligible cohort; use Study 3: Fixed 30-Day Cohort Comparison for fixed-cohort window comparisons.
Storyteller assessments: 1430
Storyteller participants: 490
ECOG records: 501
ECOG participants: 397
Storyteller features: 51
ECOG features: 12

Storyteller diagnosis distribution:
TARGET
0    1073
1     357
Name: count, dtype: int64

RUNNING WINDOW: 30_days

Window cohort:
Assessments: 77
Participants: 71
CN assessments: 53
MCI assessments: 24
Participant counts by class: {0: 49, 1: 22}

Temporal audit:
window: 30_days
window_days: 30
assessments: 77
participants: 71
CN_assessments: 53
MCI_assessments: 24
mean_ECOG_records: 1.0
median_latest_gap_days: 16.0
mean_latest_gap_days: 15.285714285714286
max_latest_gap_days: 29
Fold 1: train participants=56, test participants=15
Fold 2: train participants=56, test participants=15
Fold 3: train participants=58, test participants=13
Fold 4: t

,window,window_days,assessments,participants,CN_assessments,MCI_assessments,mean_ECOG_records,median_latest_gap_days,mean_latest_gap_days,max_latest_gap_days
0,30_days,30,77,71,53,24,1.000,16.0,15.286,29
1,90_days,90,150,132,102,48,1.000,28.0,32.840,89
2,180_days,180,183,160,121,62,1.000,35.0,50.743,179
3,365_days,365,328,175,222,106,1.003,119.5,132.454,363
4,historical,Any historical,515,185,382,133,1.066,222.0,266.955,988



Temporal Window Sensitivity Analysis ASSESSMENT-LEVEL RESULTS


,Window,Model,Assessments,Participants,Accuracy,Balanced_Accuracy,MCI_Precision,MCI_Recall,MCI_F1,Macro_F1,ROC_AUC,TN,FP,FN,TP
7,180_days,ECOG,183,160,0.6667,0.6339,0.5077,0.5323,0.5197,0.6322,0.6498,89,32,29,33
8,180_days,Multimodal,183,160,0.6831,0.6660,0.5278,0.6129,0.5672,0.6586,0.6478,87,34,24,38
6,180_days,Storyteller,183,160,0.5792,0.5363,0.3846,0.4032,0.3937,0.5358,0.5336,81,40,37,25
1,30_days,ECOG,77,71,0.7532,0.7068,0.6087,0.5833,0.5957,0.7091,0.6910,44,9,10,14
2,30_days,Multimodal,77,71,0.7662,0.7276,0.6250,0.6250,0.6250,0.7276,0.8082,44,9,9,15
0,30_days,Storyteller,77,71,0.6883,0.6254,0.5000,0.4583,0.4783,0.6280,0.6675,42,11,13,11
10,365_days,ECOG,328,175,0.6738,0.6259,0.4952,0.4906,0.4929,0.6262,0.6656,169,53,54,52
11,365_days,Multimodal,328,175,0.6860,0.6472,0.5135,0.5377,0.5253,0.6454,0.6621,168,54,49,57
9,365_days,Storyteller,328,175,0.6098,0.5811,0.4141,0.5000,0.4530,0.5748,0.5958,147,75,53,53
4,90_days,ECOG,150,132,0.7400,0.6930,0.6000,0.5625,0.5806,0.6961,0.7411,84,18,21,27



Temporal Window Sensitivity Analysis PARTICIPANT-LEVEL RESULTS


,Window,Model,Participants,Excluded_inconsistent_label_participants,Accuracy,Balanced_Accuracy,MCI_Precision,MCI_Recall,MCI_F1,Macro_F1,ROC_AUC,TN,FP,FN,TP
7,180_days,ECOG,160,0,0.6750,0.6429,0.4655,0.5625,0.5094,0.6332,0.6437,81,31,21,27
8,180_days,Multimodal,160,0,0.6875,0.6696,0.4839,0.6250,0.5455,0.6537,0.6440,80,32,18,30
6,180_days,Storyteller,160,0,0.6062,0.5580,0.3684,0.4375,0.4000,0.5535,0.5441,76,36,27,21
1,30_days,ECOG,71,0,0.7465,0.7036,0.5909,0.5909,0.5909,0.7036,0.6883,40,9,9,13
2,30_days,Multimodal,71,0,0.7746,0.7491,0.6250,0.6818,0.6522,0.7428,0.8117,40,9,7,15
0,30_days,Storyteller,71,0,0.6761,0.6150,0.4762,0.4545,0.4651,0.6164,0.6725,38,11,12,10
10,365_days,ECOG,175,0,0.6743,0.6260,0.4737,0.5000,0.4865,0.6240,0.6499,91,30,27,27
11,365_days,Multimodal,175,0,0.7029,0.6518,0.5185,0.5185,0.5185,0.6518,0.6763,95,26,26,28
9,365_days,Storyteller,175,0,0.6114,0.5857,0.4000,0.5185,0.4516,0.5754,0.6457,79,42,26,28
4,90_days,ECOG,132,0,0.7273,0.6725,0.5385,0.5385,0.5385,0.6725,0.7590,75,18,18,21



Study 2: Temporal Window Sensitivity completed.
Results saved to: C:\Users\amnar\Documents\ADNI4_Storyteller_ECOG_Classification\results\temporal_window_sensitivity


## Study 3 — Fixed 30-Day Cohort Temporal Comparison

Holds the cohort fixed at the 77 assessments / 71 participants with ECog-12 within 30 days and compares temporal aggregation windows on identical participant folds.


In [51]:
# Fixed Cohort Temporal Window Comparison COMMON-COHORT TEMPORAL COMPARISON
# ============================================================
#
# Objective:
# Compare different ECOG temporal windows using exactly the
# same Storyteller assessments and participants.
#
# Fixed cohort:
# Storyteller assessments having at least one ECOG record
# within 30 days before/on the Storyteller assessment date.
#
# ECOG windows:
# 30, 90, 180, 365 days, and all historical records.
#
# Models:
# 1. Storyteller only
# 2. ECOG only
# 3. Storyteller + ECOG fusion
#
# Evaluation:
# - 5-fold StratifiedGroupKFold
# - Grouped by participant
# - Identical folds for every comparison
# - Out-of-fold predictions
# - Assessment-level and participant-level metrics
#
# Important:
# This is a sensitivity experiment on a fixed cohort.
# It does not estimate performance for the entire ADNI cohort.
# ============================================================

import numpy as np
import pandas as pd
import warnings

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

WINDOWS = {
    "30_days": 30,
    "90_days": 90,
    "180_days": 180,
    "365_days": 365,
    "historical": None
}

N_SPLITS = 5
RANDOM_STATE = 42

STORY_DATE = "EXAMDATE_STORY"
ECOG_DATE = "COLDATE"
ID_COL = "PTID"
TARGET_COL = "TARGET"

# ------------------------------------------------------------
# 2. PREPARE DATA
# ------------------------------------------------------------

story = dataset1_storyteller.copy()
ecog = dataset2_ecog.copy()

story[STORY_DATE] = pd.to_datetime(
    story[STORY_DATE], errors="coerce"
)

ecog[ECOG_DATE] = pd.to_datetime(
    ecog[ECOG_DATE], errors="coerce"
)

story = story.dropna(
    subset=[ID_COL, STORY_DATE, TARGET_COL]
).copy()

ecog = ecog.dropna(
    subset=[ID_COL, ECOG_DATE]
).copy()

# Ensure target is numeric
story[TARGET_COL] = pd.to_numeric(
    story[TARGET_COL], errors="coerce"
)

story = story[
    story[TARGET_COL].isin([0, 1])
].copy()

# ECOG value 9 means missing
for col in ecog_feature_cols:
    ecog[col] = pd.to_numeric(
        ecog[col], errors="coerce"
    )
    ecog[col] = ecog[col].replace(9, np.nan)

# Remove ECOG records with no observed questionnaire items
ecog = ecog[
    ecog[ecog_feature_cols].notna().any(axis=1)
].copy()

# Ensure Storyteller predictors are numeric
for col in story_feature_cols:
    story[col] = pd.to_numeric(
        story[col], errors="coerce"
    )

# Reset indexes
story = story.reset_index(drop=True)
ecog = ecog.reset_index(drop=True)

print("Storyteller records:", len(story))
print("Storyteller participants:", story[ID_COL].nunique())

print("ECOG records:", len(ecog))
print("ECOG participants:", ecog[ID_COL].nunique())

# ------------------------------------------------------------
# 3. IDENTIFY THE FIXED 30-DAY COHORT
# ------------------------------------------------------------

# For each Storyteller assessment, identify ECOG records
# from the same participant dated on/before Storyteller.
#
# Only assessments with at least one ECOG record within
# 30 days are retained in the common cohort.

ecog_by_id = {
    pid: group.sort_values(ECOG_DATE).copy()
    for pid, group in ecog.groupby(ID_COL)
}

eligible_indices = []

for idx, row in story.iterrows():

    pid = row[ID_COL]
    story_date = row[STORY_DATE]

    if pid not in ecog_by_id:
        continue

    participant_ecog = ecog_by_id[pid]

    eligible = participant_ecog[
        (participant_ecog[ECOG_DATE] <= story_date) &
        (
            (story_date - participant_ecog[ECOG_DATE])
            .dt.days <= 30
        )
    ]

    if len(eligible) > 0:
        eligible_indices.append(idx)

common_story = story.loc[eligible_indices].copy()
common_story = common_story.reset_index(drop=True)

print("\nFIXED COMMON COHORT")
print("-------------------")
print("Assessments:", len(common_story))
print("Participants:", common_story[ID_COL].nunique())
print(
    "CN assessments:",
    int((common_story[TARGET_COL] == 0).sum())
)
print(
    "MCI assessments:",
    int((common_story[TARGET_COL] == 1).sum())
)

# ------------------------------------------------------------
# 4. BUILD ECOG FEATURES FOR EACH WINDOW
# ------------------------------------------------------------

# For every fixed Storyteller assessment:
# - Select same-participant ECOG records
# - Keep records dated on/before Storyteller
# - Apply the relevant maximum time gap
# - Aggregate ECOG questionnaire items using median
#
# The Storyteller assessments remain identical across windows.

window_ecog_data = {}
window_audit = []

for window_name, max_days in WINDOWS.items():

    records = []

    for _, row in common_story.iterrows():

        pid = row[ID_COL]
        story_date = row[STORY_DATE]

        participant_ecog = ecog_by_id.get(pid)

        if participant_ecog is None:
            continue

        eligible = participant_ecog[
            participant_ecog[ECOG_DATE] <= story_date
        ].copy()

        eligible["GAP_DAYS"] = (
            story_date - eligible[ECOG_DATE]
        ).dt.days

        if max_days is not None:
            eligible = eligible[
                eligible["GAP_DAYS"] <= max_days
            ]

        if eligible.empty:
            continue

        # Aggregate ECOG questionnaire items
        aggregated = eligible[ecog_feature_cols].median()

        record = {
            "ROW_ID": row.name,
            ID_COL: pid,
            TARGET_COL: row[TARGET_COL],
            "N_ECOG_RECORDS": len(eligible),
            "LATEST_ECOG_GAP": eligible["GAP_DAYS"].min(),
            "OLDEST_ECOG_GAP": eligible["GAP_DAYS"].max()
        }

        record.update(aggregated.to_dict())

        records.append(record)

    window_df = pd.DataFrame(records)

    window_ecog_data[window_name] = window_df

    window_audit.append({
        "Window": window_name,
        "Assessments": len(window_df),
        "Participants": window_df[ID_COL].nunique(),
        "Mean_ECOG_Records": (
            window_df["N_ECOG_RECORDS"].mean()
            if len(window_df) else np.nan
        ),
        "Median_Latest_Gap_Days": (
            window_df["LATEST_ECOG_GAP"].median()
            if len(window_df) else np.nan
        ),
        "Mean_Latest_Gap_Days": (
            window_df["LATEST_ECOG_GAP"].mean()
            if len(window_df) else np.nan
        ),
        "Maximum_Oldest_Gap_Days": (
            window_df["OLDEST_ECOG_GAP"].max()
            if len(window_df) else np.nan
        )
    })

window_audit_df = pd.DataFrame(window_audit)

print("\nTEMPORAL WINDOW AUDIT")
display(window_audit_df.round(3))

# ------------------------------------------------------------
# 5. VERIFY THAT THE COHORT IS IDENTICAL
# ------------------------------------------------------------

for window_name, window_df in window_ecog_data.items():

    assert len(window_df) == len(common_story), (
        f"Assessment mismatch in {window_name}"
    )

    assert set(window_df[ID_COL]) == set(common_story[ID_COL]), (
        f"Participant mismatch in {window_name}"
    )

    assert window_df["ROW_ID"].tolist() == list(common_story.index), (
        f"Row order mismatch in {window_name}"
    )

print("\nAll temporal windows use the same assessment cohort.")
print(
    "Primary window comparison: compare these results within this fixed "
    "30-day-eligible cohort. If 30/90/180-day ECOG summaries are identical, "
    "that means no additional prior ECOG records entered those wider windows "
    "for the retained assessments."
)

# ------------------------------------------------------------
# 6. CREATE IDENTICAL GROUPED CROSS-VALIDATION FOLDS
# ------------------------------------------------------------

X_story = common_story[story_feature_cols].copy()
y = common_story[TARGET_COL].astype(int).to_numpy()
groups = common_story[ID_COL].to_numpy()

cv = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Generate folds once and reuse them for every model/window.
folds = list(cv.split(X_story, y, groups))

# Check participant leakage
for fold_number, (train_idx, test_idx) in enumerate(folds, 1):

    train_groups = set(groups[train_idx])
    test_groups = set(groups[test_idx])

    assert train_groups.isdisjoint(test_groups), (
        f"Participant leakage in fold {fold_number}"
    )

print("Participant leakage check passed.")

# ------------------------------------------------------------
# 7. MODEL PIPELINE
# ------------------------------------------------------------

def make_experiment19_model():

    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
                random_state=RANDOM_STATE
            )
        )
    ])

# ------------------------------------------------------------
# 8. METRIC FUNCTION
# ------------------------------------------------------------

def calculate_experiment19_metrics(y_true, probabilities):

    predictions = (probabilities >= 0.50).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "Accuracy": accuracy_score(y_true, predictions),
        "Balanced_Accuracy": balanced_accuracy_score(
            y_true, predictions
        ),
        "MCI_Precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "MCI_Recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "MCI_F1": f1_score(
            y_true, predictions, zero_division=0
        ),
        "Macro_F1": f1_score(
            y_true, predictions, average="macro",
            zero_division=0
        ),
        "ROC_AUC": (
            roc_auc_score(y_true, probabilities)
            if len(np.unique(y_true)) == 2
            else np.nan
        ),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

# ------------------------------------------------------------
# 9. RUN ALL EXPERIMENTS USING IDENTICAL FOLDS
# ------------------------------------------------------------

experiment19_oof = {}
experiment19_results = []

for window_name, window_df in window_ecog_data.items():

    print(f"\nRunning window: {window_name}")

    X_ecog = window_df[ecog_feature_cols].copy()

    X_fusion = pd.concat(
        [
            common_story[story_feature_cols].reset_index(drop=True),
            X_ecog.reset_index(drop=True)
        ],
        axis=1
    )

    model_inputs = {
        "Storyteller": X_story,
        "ECOG": X_ecog,
        "Fusion": X_fusion
    }

    for model_name, X in model_inputs.items():

        oof_probabilities = np.full(len(y), np.nan)

        for fold_number, (train_idx, test_idx) in enumerate(
            folds, 1
        ):

            model = make_experiment19_model()

            model.fit(
                X.iloc[train_idx],
                y[train_idx]
            )

            probabilities = model.predict_proba(
                X.iloc[test_idx]
            )[:, 1]

            oof_probabilities[test_idx] = probabilities

        assert not np.isnan(oof_probabilities).any()

        experiment19_oof[
            (window_name, model_name)
        ] = oof_probabilities

        metrics = calculate_experiment19_metrics(
            y,
            oof_probabilities
        )

        experiment19_results.append({
            "Window": window_name,
            "Model": model_name,
            "Evaluation": "Assessment-level",
            "N": len(y),
            **metrics
        })

        print(
            f"{model_name}: "
            f"AUC={metrics['ROC_AUC']:.4f}, "
            f"Balanced Acc={metrics['Balanced_Accuracy']:.4f}, "
            f"MCI Recall={metrics['MCI_Recall']:.4f}"
        )

# ------------------------------------------------------------
# 10. PARTICIPANT-LEVEL EVALUATION
# ------------------------------------------------------------

# Average assessment probabilities within each participant.
# All participants in this fixed cohort should have consistent
# diagnosis labels. Verify before evaluating.

participant_labels = (
    common_story.groupby(ID_COL)[TARGET_COL]
    .nunique()
)

inconsistent_ids = participant_labels[
    participant_labels > 1
].index.tolist()

print(
    "\nParticipants with changing labels:",
    len(inconsistent_ids)
)

consistent_ids = participant_labels[
    participant_labels == 1
].index

participant_results = []

for window_name in WINDOWS:

    for model_name in ["Storyteller", "ECOG", "Fusion"]:

        probabilities = experiment19_oof[
            (window_name, model_name)
        ]

        assessment_predictions = pd.DataFrame({
            ID_COL: groups,
            TARGET_COL: y,
            "PROBABILITY": probabilities
        })

        participant_predictions = (
            assessment_predictions[
                assessment_predictions[ID_COL].isin(
                    consistent_ids
                )
            ]
            .groupby(ID_COL)
            .agg({
                TARGET_COL: "first",
                "PROBABILITY": "mean"
            })
            .reset_index()
        )

        participant_metrics = calculate_experiment19_metrics(
            participant_predictions[TARGET_COL].to_numpy(),
            participant_predictions["PROBABILITY"].to_numpy()
        )

        participant_results.append({
            "Window": window_name,
            "Model": model_name,
            "Evaluation": "Participant-level",
            "N": len(participant_predictions),
            **participant_metrics
        })

# ------------------------------------------------------------
# 11. DISPLAY RESULTS
# ------------------------------------------------------------

experiment19_results_df = pd.DataFrame(
    experiment19_results
)

experiment19_participant_df = pd.DataFrame(
    participant_results
)

metric_columns = [
    "Accuracy",
    "Balanced_Accuracy",
    "MCI_Precision",
    "MCI_Recall",
    "MCI_F1",
    "Macro_F1",
    "ROC_AUC"
]

print("\nASSESSMENT-LEVEL RESULTS")
display(
    experiment19_results_df[
        ["Window", "Model", "N"] + metric_columns
    ].round(4)
)

print("\nPARTICIPANT-LEVEL RESULTS")
display(
    experiment19_participant_df[
        ["Window", "Model", "N"] + metric_columns
    ].round(4)
)

# ------------------------------------------------------------
# 12. CALCULATE FUSION AUC GAIN OVER STORYTELLER
# ------------------------------------------------------------

participant_auc = experiment19_participant_df.pivot(
    index="Window",
    columns="Model",
    values="ROC_AUC"
)

participant_auc["Fusion_minus_Story"] = (
    participant_auc["Fusion"] -
    participant_auc["Storyteller"]
)

participant_auc["Fusion_minus_ECOG"] = (
    participant_auc["Fusion"] -
    participant_auc["ECOG"]
)

print("\nPARTICIPANT-LEVEL AUC COMPARISON")
display(participant_auc.round(4))

# ------------------------------------------------------------
# 13. SAVE RESULTS
# ------------------------------------------------------------

from pathlib import Path

output_dir = RESULTS_ROOT / "common_cohort_temporal_comparison"

output_dir.mkdir(parents=True, exist_ok=True)

window_audit_df.to_csv(
    output_dir / "common_cohort_temporal_comparison_temporal_audit.csv",
    index=False
)

experiment19_results_df.to_csv(
    output_dir / "common_cohort_temporal_comparison_assessment_results.csv",
    index=False
)

experiment19_participant_df.to_csv(
    output_dir / "common_cohort_temporal_comparison_participant_results.csv",
    index=False
)

participant_auc.to_csv(
    output_dir / "common_cohort_temporal_comparison_auc_comparison.csv"
)

print("\nStudy 3: Fixed 30-Day Cohort Comparison completed.")
print("Results saved to:", output_dir)



# ============================================================


Storyteller records: 1430
Storyteller participants: 490
ECOG records: 501
ECOG participants: 397

FIXED COMMON COHORT
-------------------
Assessments: 77
Participants: 71
CN assessments: 53
MCI assessments: 24

TEMPORAL WINDOW AUDIT


,Window,Assessments,Participants,Mean_ECOG_Records,Median_Latest_Gap_Days,Mean_Latest_Gap_Days,Maximum_Oldest_Gap_Days
0,30_days,77,71,1.000,16.0,15.286,29
1,90_days,77,71,1.000,16.0,15.286,29
2,180_days,77,71,1.000,16.0,15.286,29
3,365_days,77,71,1.013,16.0,15.286,299
4,historical,77,71,1.130,16.0,15.286,766



All temporal windows use the same assessment cohort.
Primary window comparison: compare these results within this fixed 30-day-eligible cohort. If 30/90/180-day ECOG summaries are identical, that means no additional prior ECOG records entered those wider windows for the retained assessments.
Participant leakage check passed.

Running window: 30_days
Storyteller: AUC=0.6675, Balanced Acc=0.6254, MCI Recall=0.4583
ECOG: AUC=0.6910, Balanced Acc=0.7068, MCI Recall=0.5833
Fusion: AUC=0.8082, Balanced Acc=0.7276, MCI Recall=0.6250

Running window: 90_days
Storyteller: AUC=0.6675, Balanced Acc=0.6254, MCI Recall=0.4583
ECOG: AUC=0.6910, Balanced Acc=0.7068, MCI Recall=0.5833
Fusion: AUC=0.8082, Balanced Acc=0.7276, MCI Recall=0.6250

Running window: 180_days
Storyteller: AUC=0.6675, Balanced Acc=0.6254, MCI Recall=0.4583
ECOG: AUC=0.6910, Balanced Acc=0.7068, MCI Recall=0.5833
Fusion: AUC=0.8082, Balanced Acc=0.7276, MCI Recall=0.6250

Running window: 365_days
Storyteller: AUC=0.6675, Balan

,Window,Model,N,Accuracy,Balanced_Accuracy,MCI_Precision,MCI_Recall,MCI_F1,Macro_F1,ROC_AUC
0,30_days,Storyteller,77,0.6883,0.6254,0.5000,0.4583,0.4783,0.6280,0.6675
1,30_days,ECOG,77,0.7532,0.7068,0.6087,0.5833,0.5957,0.7091,0.6910
2,30_days,Fusion,77,0.7662,0.7276,0.6250,0.6250,0.6250,0.7276,0.8082
3,90_days,Storyteller,77,0.6883,0.6254,0.5000,0.4583,0.4783,0.6280,0.6675
4,90_days,ECOG,77,0.7532,0.7068,0.6087,0.5833,0.5957,0.7091,0.6910
5,90_days,Fusion,77,0.7662,0.7276,0.6250,0.6250,0.6250,0.7276,0.8082
6,180_days,Storyteller,77,0.6883,0.6254,0.5000,0.4583,0.4783,0.6280,0.6675
7,180_days,ECOG,77,0.7532,0.7068,0.6087,0.5833,0.5957,0.7091,0.6910
8,180_days,Fusion,77,0.7662,0.7276,0.6250,0.6250,0.6250,0.7276,0.8082
9,365_days,Storyteller,77,0.6883,0.6254,0.5000,0.4583,0.4783,0.6280,0.6675



PARTICIPANT-LEVEL RESULTS


,Window,Model,N,Accuracy,Balanced_Accuracy,MCI_Precision,MCI_Recall,MCI_F1,Macro_F1,ROC_AUC
0,30_days,Storyteller,71,0.6761,0.6150,0.4762,0.4545,0.4651,0.6164,0.6725
1,30_days,ECOG,71,0.7465,0.7036,0.5909,0.5909,0.5909,0.7036,0.6883
2,30_days,Fusion,71,0.7746,0.7491,0.6250,0.6818,0.6522,0.7428,0.8117
3,90_days,Storyteller,71,0.6761,0.6150,0.4762,0.4545,0.4651,0.6164,0.6725
4,90_days,ECOG,71,0.7465,0.7036,0.5909,0.5909,0.5909,0.7036,0.6883
5,90_days,Fusion,71,0.7746,0.7491,0.6250,0.6818,0.6522,0.7428,0.8117
6,180_days,Storyteller,71,0.6761,0.6150,0.4762,0.4545,0.4651,0.6164,0.6725
7,180_days,ECOG,71,0.7465,0.7036,0.5909,0.5909,0.5909,0.7036,0.6883
8,180_days,Fusion,71,0.7746,0.7491,0.6250,0.6818,0.6522,0.7428,0.8117
9,365_days,Storyteller,71,0.6761,0.6150,0.4762,0.4545,0.4651,0.6164,0.6725



PARTICIPANT-LEVEL AUC COMPARISON


Model,ECOG,Fusion,Storyteller,Fusion_minus_Story,Fusion_minus_ECOG
Window,,,,,
180_days,0.6883,0.8117,0.6725,0.1391,0.1234
30_days,0.6883,0.8117,0.6725,0.1391,0.1234
365_days,0.6753,0.8117,0.6725,0.1391,0.1364
90_days,0.6883,0.8117,0.6725,0.1391,0.1234
historical,0.6642,0.7894,0.6725,0.1169,0.1252



Study 3: Fixed 30-Day Cohort Comparison completed.
Results saved to: c:\Users\amnar\Documents\ADNI4_Storyteller_ECOG_Classification\results\common_cohort_temporal_comparison


## Export Primary 30-Day Temporally Matched Multimodal Cohort

Saves the canonical 77-assessment / 71-participant cohort built in Study 3 to `data/processed/primary_multimodal_30day_cohort.csv` so all downstream evaluation and SHAP analyses (`04` and `05`) load this single dataset.


In [52]:
# Export the single primary 30-day temporally matched multimodal cohort
# (77 assessments / 71 participants) used across Studies 4-13.

story_30d = common_story.copy().reset_index(drop=True)
ecog_30d = window_ecog_data["30_days"].copy().reset_index(drop=True)

if len(story_30d) != len(ecog_30d):
    raise ValueError("Storyteller and 30-day ECOG row counts do not match.")
if "ROW_ID" in ecog_30d.columns and ecog_30d["ROW_ID"].tolist() != list(range(len(story_30d))):
    raise ValueError("30-day ECOG ROW_ID alignment does not match common_story.")

primary_multimodal_30day_cohort = pd.concat(
    [
        ecog_30d[["ROW_ID", ID_COL, TARGET_COL, "N_ECOG_RECORDS", "LATEST_ECOG_GAP", "OLDEST_ECOG_GAP"]].reset_index(drop=True),
        story_30d[[STORY_DATE]].reset_index(drop=True),
        story_30d[story_feature_cols].apply(pd.to_numeric, errors="coerce").reset_index(drop=True),
        ecog_30d[ecog_feature_cols].apply(pd.to_numeric, errors="coerce").reset_index(drop=True),
    ],
    axis=1,
)

primary_multimodal_30day_cohort.to_csv(PRIMARY_COHORT_CSV, index=False)

print("Saved primary 30-day multimodal cohort to:", PRIMARY_COHORT_CSV)
print(
    f"Shape: {primary_multimodal_30day_cohort.shape} | "
    f"Assessments: {len(primary_multimodal_30day_cohort)} | "
    f"Participants: {primary_multimodal_30day_cohort[ID_COL].nunique()} | "
    f"CN: {int((primary_multimodal_30day_cohort[TARGET_COL] == 0).sum())}, "
    f"MCI: {int((primary_multimodal_30day_cohort[TARGET_COL] == 1).sum())}"
)


Saved primary 30-day multimodal cohort to: c:\Users\amnar\Documents\ADNI4_Storyteller_ECOG_Classification\data\processed\primary_multimodal_30day_cohort.csv
Shape: (77, 70) | Assessments: 77 | Participants: 71 | CN: 53, MCI: 24


## Study 4 — Leave-One-Item-Out ECog-12 Ablation

Performs leave-one-item-out ablation across all 12 ECog-12 PT items on the fixed 30-day primary cohort.


In [53]:
# ECOG Item Ablation Analysis ECOG FEATURE ABLATION ANALYSIS
# ============================================================
#
# Objective:
# Identify the contribution of individual ECOG questionnaire
# items to Storyteller + ECOG multimodal CN-MCI classification.
#
# Fixed cohort:
# 77 assessments / 71 participants (Study 3: Fixed 30-Day Cohort Comparison, 30-day cohort)
#
# Models:
# 1. Storyteller only
# 2. ECOG only
# 3. Full Storyteller + ECOG fusion
# 4. Fusion without ECOGPT1
# 5. Fusion without ECOGPT2
# ...
# 15. Fusion without ECOGPT12
#
# Evaluation:
# - Same five grouped CV folds as Study 3: Fixed 30-Day Cohort Comparison
# - Grouped by participant
# - Identical train/test splits
# - Out-of-fold predictions
# - Assessment-level metrics
# - Participant-level metrics
#
# Important:
# This is a leave-one-feature-out ablation.
# It measures the effect of removing each item individually.
# It does not establish causal importance.
# ============================================================

import numpy as np
import pandas as pd
import warnings

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from pathlib import Path

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. VERIFY STUDY 3: FIXED 30-DAY COHORT COMPARISON VARIABLES
# ------------------------------------------------------------

required_variables = [
    "common_story",
    "window_ecog_data",
    "story_feature_cols",
    "ecog_feature_cols",
    "folds",
    "groups",
    "y",
    "ID_COL",
    "TARGET_COL"
]

missing_variables = [
    variable for variable in required_variables
    if variable not in globals()
]

if missing_variables:
    raise ValueError(
        "Missing required Study 3: Fixed 30-Day Cohort Comparison variables: "
        + str(missing_variables)
    )

# Use the exact 30-day ECOG data from Study 3: Fixed 30-Day Cohort Comparison.
ecog_30 = window_ecog_data["30_days"].copy()

# Ensure the fixed assessment ordering is preserved.
assert len(common_story) == 77
assert common_story[ID_COL].nunique() == 71

assert len(ecog_30) == len(common_story)
assert ecog_30["ROW_ID"].tolist() == list(common_story.index)

# Check that the same groups and labels are being used.
assert np.array_equal(
    common_story[ID_COL].to_numpy(),
    groups
)

assert np.array_equal(
    common_story[TARGET_COL].astype(int).to_numpy(),
    y
)

print("Study 4: ECOG-12 Item Ablation cohort verified.")
print("Assessments:", len(common_story))
print("Participants:", common_story[ID_COL].nunique())
print("Storyteller features:", len(story_feature_cols))
print("ECOG features:", len(ecog_feature_cols))

# ------------------------------------------------------------
# 2. PREPARE FEATURE MATRICES
# ------------------------------------------------------------

X_story20 = (
    common_story[story_feature_cols]
    .reset_index(drop=True)
    .copy()
)

X_ecog20 = (
    ecog_30[ecog_feature_cols]
    .reset_index(drop=True)
    .copy()
)

X_fusion20 = pd.concat(
    [X_story20, X_ecog20],
    axis=1
)

y20 = common_story[TARGET_COL].astype(int).to_numpy()
groups20 = common_story[ID_COL].to_numpy()

print("\nFeature matrix dimensions:")
print("Storyteller:", X_story20.shape)
print("ECOG:", X_ecog20.shape)
print("Full fusion:", X_fusion20.shape)

# ------------------------------------------------------------
# 3. VERIFY IDENTICAL PARTICIPANT-GROUPED FOLDS
# ------------------------------------------------------------

# Reuse the exact folds generated in Study 3: Fixed 30-Day Cohort Comparison.
# Do not regenerate folds for individual ablation models.

for fold_number, (train_idx, test_idx) in enumerate(folds, 1):

    train_participants = set(groups20[train_idx])
    test_participants = set(groups20[test_idx])

    assert train_participants.isdisjoint(test_participants)

print("\nParticipant leakage check passed.")
print("Number of folds:", len(folds))

# ------------------------------------------------------------
# 4. MODEL PIPELINE
# ------------------------------------------------------------

def make_experiment20_model():

    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
                random_state=42
            )
        )
    ])

# ------------------------------------------------------------
# 5. METRIC FUNCTION
# ------------------------------------------------------------

def calculate_experiment20_metrics(
    y_true,
    probabilities
):

    predictions = (
        np.asarray(probabilities) >= 0.50
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "Accuracy": accuracy_score(
            y_true, predictions
        ),

        "Balanced_Accuracy": balanced_accuracy_score(
            y_true, predictions
        ),

        "MCI_Precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "MCI_Recall": recall_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "MCI_F1": f1_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "Macro_F1": f1_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0
        ),

        "ROC_AUC": (
            roc_auc_score(y_true, probabilities)
            if len(np.unique(y_true)) == 2
            else np.nan
        ),

        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

# ------------------------------------------------------------
# 6. DEFINE ALL ABLATION EXPERIMENTS
# ------------------------------------------------------------

ablation_models = {
    "Storyteller_only": X_story20,
    "ECOG_only": X_ecog20,
    "Full_fusion": X_fusion20
}

# Remove one ECOG questionnaire item at a time.
for ecog_item in ecog_feature_cols:

    remaining_ecog = [
        col for col in ecog_feature_cols
        if col != ecog_item
    ]

    X_ablation = pd.concat(
        [
            X_story20.reset_index(drop=True),
            X_ecog20[remaining_ecog].reset_index(drop=True)
        ],
        axis=1
    )

    model_name = f"Fusion_without_{ecog_item}"

    ablation_models[model_name] = X_ablation

print("\nModels to evaluate:")
for model_name, X in ablation_models.items():
    print(f"{model_name}: {X.shape[1]} features")

# ------------------------------------------------------------
# 7. RUN GROUPED CROSS-VALIDATION
# ------------------------------------------------------------

experiment20_oof = {}
experiment20_assessment_results = []

for model_name, X in ablation_models.items():

    print(f"\nRunning: {model_name}")

    oof_probabilities = np.full(
        len(y20),
        np.nan
    )

    for fold_number, (train_idx, test_idx) in enumerate(
        folds, 1
    ):

        model = make_experiment20_model()

        model.fit(
            X.iloc[train_idx],
            y20[train_idx]
        )

        probabilities = model.predict_proba(
            X.iloc[test_idx]
        )[:, 1]

        oof_probabilities[test_idx] = probabilities

    assert not np.isnan(oof_probabilities).any()

    experiment20_oof[model_name] = oof_probabilities

    metrics = calculate_experiment20_metrics(
        y20,
        oof_probabilities
    )

    experiment20_assessment_results.append({
        "Model": model_name,
        "Evaluation": "Assessment-level",
        "N": len(y20),
        "Features": X.shape[1],
        **metrics
    })

    print(
        f"AUC: {metrics['ROC_AUC']:.4f} | "
        f"Balanced accuracy: "
        f"{metrics['Balanced_Accuracy']:.4f} | "
        f"MCI recall: {metrics['MCI_Recall']:.4f}"
    )

# ------------------------------------------------------------
# 8. PARTICIPANT-LEVEL EVALUATION
# ------------------------------------------------------------

# Average out-of-fold assessment probabilities within
# each participant.

participant_label_counts = (
    common_story.groupby(ID_COL)[TARGET_COL]
    .nunique()
)

inconsistent_participants = (
    participant_label_counts[
        participant_label_counts > 1
    ].index
)

consistent_participants = (
    participant_label_counts[
        participant_label_counts == 1
    ].index
)

print(
    "\nParticipants with changing labels:",
    len(inconsistent_participants)
)

experiment20_participant_results = []

for model_name, probabilities in experiment20_oof.items():

    prediction_df = pd.DataFrame({
        ID_COL: groups20,
        TARGET_COL: y20,
        "Probability": probabilities
    })

    prediction_df = prediction_df[
        prediction_df[ID_COL].isin(
            consistent_participants
        )
    ].copy()

    participant_predictions = (
        prediction_df
        .groupby(ID_COL)
        .agg({
            TARGET_COL: "first",
            "Probability": "mean"
        })
        .reset_index()
    )

    metrics = calculate_experiment20_metrics(
        participant_predictions[TARGET_COL].to_numpy(),
        participant_predictions["Probability"].to_numpy()
    )

    experiment20_participant_results.append({
        "Model": model_name,
        "Evaluation": "Participant-level",
        "N": len(participant_predictions),
        "Features": ablation_models[model_name].shape[1],
        **metrics
    })

# ------------------------------------------------------------
# 9. DISPLAY RESULTS
# ------------------------------------------------------------

experiment20_assessment_df = pd.DataFrame(
    experiment20_assessment_results
)

experiment20_participant_df = pd.DataFrame(
    experiment20_participant_results
)

metric_columns = [
    "Accuracy",
    "Balanced_Accuracy",
    "MCI_Precision",
    "MCI_Recall",
    "MCI_F1",
    "Macro_F1",
    "ROC_AUC"
]

print("\nASSESSMENT-LEVEL ABLATION RESULTS")

display(
    experiment20_assessment_df[
        ["Model", "N", "Features"] + metric_columns
    ].round(4)
)

print("\nPARTICIPANT-LEVEL ABLATION RESULTS")

display(
    experiment20_participant_df[
        ["Model", "N", "Features"] + metric_columns
    ].round(4)
)

# ------------------------------------------------------------
# 10. CALCULATE AUC CHANGE AFTER REMOVING EACH ECOG ITEM
# ------------------------------------------------------------

participant_auc = (
    experiment20_participant_df
    .set_index("Model")["ROC_AUC"]
)

full_fusion_auc = participant_auc["Full_fusion"]

ablation_comparison = []

for ecog_item in ecog_feature_cols:

    model_name = f"Fusion_without_{ecog_item}"

    ablated_auc = participant_auc[model_name]

    ablation_comparison.append({
        "Removed_ECOG_Item": ecog_item,
        "Full_Fusion_AUC": full_fusion_auc,
        "Ablated_Fusion_AUC": ablated_auc,

        # Positive means removing the item reduced AUC.
        # Negative means removing the item increased AUC.
        "AUC_Drop_After_Removal": (
            full_fusion_auc - ablated_auc
        )
    })

ablation_comparison_df = pd.DataFrame(
    ablation_comparison
)

ablation_comparison_df = (
    ablation_comparison_df
    .sort_values(
        "AUC_Drop_After_Removal",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nECOG ITEM ABLATION CONTRIBUTION")
display(
    ablation_comparison_df.round(4)
)

# ------------------------------------------------------------
# 11. COMPARE THE BEST AND WORST ABLATIONS
# ------------------------------------------------------------

print("\nABLATION SUMMARY")

print(
    "Full fusion participant-level AUC:",
    round(full_fusion_auc, 4)
)

best_ablation = ablation_comparison_df.iloc[0]
worst_ablation = ablation_comparison_df.iloc[-1]

print(
    "\nLargest AUC reduction after removal:",
    best_ablation["Removed_ECOG_Item"],
    " | AUC drop:",
    round(best_ablation["AUC_Drop_After_Removal"], 4)
)

print(
    "\nLargest AUC improvement after removal:",
    worst_ablation["Removed_ECOG_Item"],
    " | AUC change:",
    round(-worst_ablation["AUC_Drop_After_Removal"], 4)
)

# ------------------------------------------------------------
# 12. SAVE ALL RESULTS
# ------------------------------------------------------------

output_dir = RESULTS_ROOT / "ecog_item_ablation"

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

experiment20_assessment_df.to_csv(
    output_dir / "ecog_item_ablation_assessment_ablation.csv",
    index=False
)

experiment20_participant_df.to_csv(
    output_dir / "ecog_item_ablation_participant_ablation.csv",
    index=False
)

ablation_comparison_df.to_csv(
    output_dir / "ecog_item_ablation_ecog_item_contribution.csv",
    index=False
)

# Save all OOF probabilities for later analysis.
oof_df = pd.DataFrame({
    ID_COL: groups20,
    TARGET_COL: y20
})

for model_name, probabilities in experiment20_oof.items():

    oof_df[model_name] = probabilities

oof_df.to_csv(
    output_dir / "ecog_item_ablation_oof_probabilities.csv",
    index=False
)

print("\n========================================")
print("STUDY 4: ECOG-12 ITEM ABLATION COMPLETED")
print("========================================")
print("Results saved to:", output_dir)



# ============================================================


Study 4: ECOG-12 Item Ablation cohort verified.
Assessments: 77
Participants: 71
Storyteller features: 51
ECOG features: 12

Feature matrix dimensions:
Storyteller: (77, 51)
ECOG: (77, 12)
Full fusion: (77, 63)

Participant leakage check passed.
Number of folds: 5

Models to evaluate:
Storyteller_only: 51 features
ECOG_only: 12 features
Full_fusion: 63 features
Fusion_without_ECOGPT1: 62 features
Fusion_without_ECOGPT2: 62 features
Fusion_without_ECOGPT3: 62 features
Fusion_without_ECOGPT4: 62 features
Fusion_without_ECOGPT5: 62 features
Fusion_without_ECOGPT6: 62 features
Fusion_without_ECOGPT7: 62 features
Fusion_without_ECOGPT8: 62 features
Fusion_without_ECOGPT9: 62 features
Fusion_without_ECOGPT10: 62 features
Fusion_without_ECOGPT11: 62 features
Fusion_without_ECOGPT12: 62 features

Running: Storyteller_only
AUC: 0.6675 | Balanced accuracy: 0.6254 | MCI recall: 0.4583

Running: ECOG_only
AUC: 0.6910 | Balanced accuracy: 0.7068 | MCI recall: 0.5833

Running: Full_fusion
AUC: 0.808

,Model,N,Features,Accuracy,Balanced_Accuracy,MCI_Precision,MCI_Recall,MCI_F1,Macro_F1,ROC_AUC
0,Storyteller_only,77,51,0.6883,0.6254,0.5000,0.4583,0.4783,0.6280,0.6675
1,ECOG_only,77,12,0.7532,0.7068,0.6087,0.5833,0.5957,0.7091,0.6910
2,Full_fusion,77,63,0.7662,0.7276,0.6250,0.6250,0.6250,0.7276,0.8082
3,Fusion_without_ECOGPT1,77,62,0.7792,0.7484,0.6400,0.6667,0.6531,0.7456,0.8255
4,Fusion_without_ECOGPT2,77,62,0.7532,0.7182,0.6000,0.6250,0.6122,0.7156,0.8522
5,Fusion_without_ECOGPT3,77,62,0.7662,0.7276,0.6250,0.6250,0.6250,0.7276,0.8278
6,Fusion_without_ECOGPT4,77,62,0.7532,0.7068,0.6087,0.5833,0.5957,0.7091,0.8035
7,Fusion_without_ECOGPT5,77,62,0.7273,0.6879,0.5600,0.5833,0.5714,0.6857,0.7917
8,Fusion_without_ECOGPT6,77,62,0.7273,0.6651,0.5714,0.5000,0.5333,0.6703,0.7956
9,Fusion_without_ECOGPT7,77,62,0.7532,0.7068,0.6087,0.5833,0.5957,0.7091,0.8105



PARTICIPANT-LEVEL ABLATION RESULTS


,Model,N,Features,Accuracy,Balanced_Accuracy,MCI_Precision,MCI_Recall,MCI_F1,Macro_F1,ROC_AUC
0,Storyteller_only,71,51,0.6761,0.6150,0.4762,0.4545,0.4651,0.6164,0.6725
1,ECOG_only,71,12,0.7465,0.7036,0.5909,0.5909,0.5909,0.7036,0.6883
2,Full_fusion,71,63,0.7746,0.7491,0.6250,0.6818,0.6522,0.7428,0.8117
3,Fusion_without_ECOGPT1,71,62,0.7887,0.7718,0.6400,0.7273,0.6809,0.7615,0.8321
4,Fusion_without_ECOGPT2,71,62,0.7606,0.7389,0.6000,0.6818,0.6383,0.7297,0.8609
5,Fusion_without_ECOGPT3,71,62,0.7746,0.7491,0.6250,0.6818,0.6522,0.7428,0.8312
6,Fusion_without_ECOGPT4,71,62,0.7606,0.7263,0.6087,0.6364,0.6222,0.7235,0.8135
7,Fusion_without_ECOGPT5,71,62,0.7324,0.7059,0.5600,0.6364,0.5957,0.6979,0.7922
8,Fusion_without_ECOGPT6,71,62,0.7324,0.6809,0.5714,0.5455,0.5581,0.6831,0.7978
9,Fusion_without_ECOGPT7,71,62,0.7606,0.7263,0.6087,0.6364,0.6222,0.7235,0.8154



ECOG ITEM ABLATION CONTRIBUTION


,Removed_ECOG_Item,Full_Fusion_AUC,Ablated_Fusion_AUC,AUC_Drop_After_Removal
0,ECOGPT9,0.8117,0.7839,0.0278
1,ECOGPT8,0.8117,0.7857,0.0260
2,ECOGPT5,0.8117,0.7922,0.0195
3,ECOGPT6,0.8117,0.7978,0.0139
4,ECOGPT10,0.8117,0.7996,0.0121
5,ECOGPT4,0.8117,0.8135,-0.0019
6,ECOGPT11,0.8117,0.8135,-0.0019
7,ECOGPT7,0.8117,0.8154,-0.0037
8,ECOGPT12,0.8117,0.8154,-0.0037
9,ECOGPT3,0.8117,0.8312,-0.0195



ABLATION SUMMARY
Full fusion participant-level AUC: 0.8117

Largest AUC reduction after removal: ECOGPT9  | AUC drop: 0.0278

Largest AUC improvement after removal: ECOGPT2  | AUC change: 0.0492

STUDY 4: ECOG-12 ITEM ABLATION COMPLETED
Results saved to: c:\Users\amnar\Documents\ADNI4_Storyteller_ECOG_Classification\results\ecog_item_ablation


## Study 5 — Repeated-Seed ECog-12 Ablation Stability

Tests whether excluding `ECOGPT1` and/or `ECOGPT2` remains stable across 10 repeated 5-fold `StratifiedGroupKFold` seeds.


In [54]:
# Repeated Seed Stability of ECOG Item Ablations STABILITY ANALYSIS OF ECOG FEATURE ABLATION
# ============================================================
#
# Objective:
# Determine whether the improvements observed in Study 4: ECOG-12 Item Ablation
# are stable across different participant-grouped CV splits.
#
# Fixed cohort:
# 77 assessments / 71 participants
#
# Models:
# 1. Storyteller only
# 2. Full fusion
# 3. Fusion without ECOGPT1
# 4. Fusion without ECOGPT2
# 5. Fusion without ECOGPT8
# 6. Fusion without ECOGPT9
#
# Evaluation:
# - 10 random seeds
# - 5 StratifiedGroupKFold folds per seed
# - Participant-level grouping
# - Same folds for all models within each seed
# - Out-of-fold probabilities
# - Assessment-level and participant-level metrics
#
# Important:
# The standard deviation across seeds measures split sensitivity.
# It is not a confidence interval or significance test.
# ============================================================

import numpy as np
import pandas as pd
import warnings

from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

N_SPLITS = 5

RANDOM_SEEDS = [
    11, 21, 42, 52, 62,
    72, 82, 92, 102, 112
]

ABLATION_ITEMS = [
    "ECOGPT1",
    "ECOGPT2",
    "ECOGPT8",
    "ECOGPT9"
]

# ------------------------------------------------------------
# 2. VERIFY STUDY 4: ECOG-12 ITEM ABLATION COHORT
# ------------------------------------------------------------

required_variables = [
    "common_story",
    "window_ecog_data",
    "story_feature_cols",
    "ecog_feature_cols",
    "ID_COL",
    "TARGET_COL"
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise ValueError(
        "Missing required variables: "
        + str(missing_variables)
    )

# Reuse the exact 30-day cohort.
common_story21 = common_story.copy().reset_index(drop=True)

ecog21 = (
    window_ecog_data["30_days"]
    .copy()
    .reset_index(drop=True)
)

assert len(common_story21) == 77
assert common_story21[ID_COL].nunique() == 71

assert len(ecog21) == len(common_story21)

assert (
    ecog21["ROW_ID"].tolist()
    == list(range(len(common_story21)))
)

# Check consistent labels within participants.
participant_label_counts = (
    common_story21.groupby(ID_COL)[TARGET_COL]
    .nunique()
)

inconsistent_ids = participant_label_counts[
    participant_label_counts > 1
].index.tolist()

if inconsistent_ids:
    raise ValueError(
        "Some participants have inconsistent labels: "
        + str(inconsistent_ids)
    )

print("Study 5: Repeated-Seed Ablation Stability cohort verified.")
print("Assessments:", len(common_story21))
print("Participants:", common_story21[ID_COL].nunique())
print(
    "CN assessments:",
    int((common_story21[TARGET_COL] == 0).sum())
)
print(
    "MCI assessments:",
    int((common_story21[TARGET_COL] == 1).sum())
)

# ------------------------------------------------------------
# 3. PREPARE FEATURE MATRICES
# ------------------------------------------------------------

X_story21 = (
    common_story21[story_feature_cols]
    .reset_index(drop=True)
    .copy()
)

X_ecog21 = (
    ecog21[ecog_feature_cols]
    .reset_index(drop=True)
    .copy()
)

y21 = (
    common_story21[TARGET_COL]
    .astype(int)
    .to_numpy()
)

groups21 = common_story21[ID_COL].to_numpy()

# Full fusion
X_full21 = pd.concat(
    [
        X_story21,
        X_ecog21
    ],
    axis=1
)

# ------------------------------------------------------------
# 4. CREATE THE SIX MODEL VARIANTS
# ------------------------------------------------------------

model_inputs21 = {
    "Storyteller_only": X_story21,
    "Full_fusion": X_full21
}

# Leave-one-item-out models for selected ECOG items.
for item in ABLATION_ITEMS:

    remaining_items = [
        col for col in ecog_feature_cols
        if col != item
    ]

    X_ablation = pd.concat(
        [
            X_story21,
            X_ecog21[remaining_items]
        ],
        axis=1
    )

    model_inputs21[
        f"Fusion_without_{item}"
    ] = X_ablation

print("\nMODELS")
for name, X in model_inputs21.items():
    print(name, X.shape)

# ------------------------------------------------------------
# 5. MODEL PIPELINE
# ------------------------------------------------------------

def make_experiment21_model():

    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
                random_state=42
            )
        )
    ])

# ------------------------------------------------------------
# 6. METRIC FUNCTION
# ------------------------------------------------------------

def calculate_experiment21_metrics(
    y_true,
    probabilities
):

    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities)

    predictions = (
        probabilities >= 0.50
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "Accuracy": accuracy_score(
            y_true, predictions
        ),

        "Balanced_Accuracy": balanced_accuracy_score(
            y_true, predictions
        ),

        "MCI_Precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "MCI_Recall": recall_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "MCI_F1": f1_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "Macro_F1": f1_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0
        ),

        "ROC_AUC": (
            roc_auc_score(y_true, probabilities)
            if len(np.unique(y_true)) == 2
            else np.nan
        ),

        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

# ------------------------------------------------------------
# 7. PREPARE PARTICIPANT LABELS
# ------------------------------------------------------------

participant_labels21 = (
    common_story21
    .groupby(ID_COL)[TARGET_COL]
    .first()
)

participant_ids21 = participant_labels21.index.to_numpy()

participant_y21 = (
    participant_labels21
    .astype(int)
    .to_numpy()
)

print("\nPARTICIPANT CLASS DISTRIBUTION")
print(
    pd.Series(participant_y21)
    .value_counts()
    .sort_index()
    .rename(index={0: "CN", 1: "MCI"})
)

# ------------------------------------------------------------
# 8. RUN REPEATED GROUPED CROSS-VALIDATION
# ------------------------------------------------------------

assessment_records21 = []
participant_records21 = []

# Save OOF predictions for every seed and model.
experiment21_oof = {}

for seed in RANDOM_SEEDS:

    print("\n" + "=" * 55)
    print("RANDOM SEED:", seed)
    print("=" * 55)

    cv = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=seed
    )

    # Generate one set of folds and reuse it for every model.
    folds21 = list(
        cv.split(
            X_story21,
            y21,
            groups21
        )
    )

    # Verify no participant leakage.
    for fold_number, (train_idx, test_idx) in enumerate(
        folds21, 1
    ):

        train_ids = set(groups21[train_idx])
        test_ids = set(groups21[test_idx])

        assert train_ids.isdisjoint(test_ids), (
            f"Participant leakage: seed={seed}, "
            f"fold={fold_number}"
        )

    # Confirm every assessment receives exactly one OOF prediction.
    for model_name, X in model_inputs21.items():

        oof_probabilities = np.full(
            len(y21),
            np.nan
        )

        for fold_number, (train_idx, test_idx) in enumerate(
            folds21, 1
        ):

            model = make_experiment21_model()

            model.fit(
                X.iloc[train_idx],
                y21[train_idx]
            )

            fold_probabilities = (
                model.predict_proba(
                    X.iloc[test_idx]
                )[:, 1]
            )

            oof_probabilities[test_idx] = (
                fold_probabilities
            )

        assert not np.isnan(
            oof_probabilities
        ).any()

        # Save assessment-level predictions.
        experiment21_oof[
            (seed, model_name)
        ] = oof_probabilities.copy()

        # Assessment-level metrics.
        assessment_metrics = (
            calculate_experiment21_metrics(
                y21,
                oof_probabilities
            )
        )

        assessment_records21.append({
            "Seed": seed,
            "Model": model_name,
            "Evaluation": "Assessment-level",
            "N": len(y21),
            **assessment_metrics
        })

        # Participant-level probability averaging.
        assessment_prediction_df = pd.DataFrame({
            ID_COL: groups21,
            TARGET_COL: y21,
            "Probability": oof_probabilities
        })

        participant_prediction_df = (
            assessment_prediction_df
            .groupby(ID_COL)
            .agg({
                TARGET_COL: "first",
                "Probability": "mean"
            })
            .reset_index()
        )

        # Verify all participants are represented.
        assert len(participant_prediction_df) == 71

        participant_metrics = (
            calculate_experiment21_metrics(
                participant_prediction_df[TARGET_COL],
                participant_prediction_df["Probability"]
            )
        )

        participant_records21.append({
            "Seed": seed,
            "Model": model_name,
            "Evaluation": "Participant-level",
            "N": len(participant_prediction_df),
            **participant_metrics
        })

        print(
            f"{model_name}: "
            f"AUC={participant_metrics['ROC_AUC']:.4f}, "
            f"Balanced Acc="
            f"{participant_metrics['Balanced_Accuracy']:.4f}, "
            f"MCI Recall="
            f"{participant_metrics['MCI_Recall']:.4f}"
        )

# ------------------------------------------------------------
# 9. CONVERT RESULTS INTO DATAFRAMES
# ------------------------------------------------------------

experiment21_assessment_df = pd.DataFrame(
    assessment_records21
)

experiment21_participant_df = pd.DataFrame(
    participant_records21
)

print("\nRepeated grouped CV completed.")
print("Seeds:", len(RANDOM_SEEDS))
print("Folds per seed:", N_SPLITS)
print(
    "Total model-fold evaluations:",
    len(RANDOM_SEEDS) * N_SPLITS * len(model_inputs21)
)

# ------------------------------------------------------------
# 10. PARTICIPANT-LEVEL MEAN AND STANDARD DEVIATION
# ------------------------------------------------------------

metric_columns21 = [
    "Accuracy",
    "Balanced_Accuracy",
    "MCI_Precision",
    "MCI_Recall",
    "MCI_F1",
    "Macro_F1",
    "ROC_AUC"
]

participant_stability21 = (
    experiment21_participant_df
    .groupby("Model")[metric_columns21]
    .agg(["mean", "std"])
)

# Flatten multi-level column names.
participant_stability21.columns = [
    f"{metric}_{stat}"
    for metric, stat in participant_stability21.columns
]

participant_stability21 = (
    participant_stability21.reset_index()
)

print("\nPARTICIPANT-LEVEL STABILITY RESULTS")
display(
    participant_stability21.round(4)
)

# ------------------------------------------------------------
# 11. ASSESSMENT-LEVEL MEAN AND STANDARD DEVIATION
# ------------------------------------------------------------

assessment_stability21 = (
    experiment21_assessment_df
    .groupby("Model")[metric_columns21]
    .agg(["mean", "std"])
)

assessment_stability21.columns = [
    f"{metric}_{stat}"
    for metric, stat in assessment_stability21.columns
]

assessment_stability21 = (
    assessment_stability21.reset_index()
)

print("\nASSESSMENT-LEVEL STABILITY RESULTS")
display(
    assessment_stability21.round(4)
)

# ------------------------------------------------------------
# 12. AUC COMPARISON AGAINST FULL FUSION
# ------------------------------------------------------------

auc_seed_table21 = (
    experiment21_participant_df
    .pivot(
        index="Seed",
        columns="Model",
        values="ROC_AUC"
    )
)

comparison_models21 = [
    "Storyteller_only",
    "Fusion_without_ECOGPT1",
    "Fusion_without_ECOGPT2",
    "Fusion_without_ECOGPT8",
    "Fusion_without_ECOGPT9"
]

auc_difference_records21 = []

for model_name in comparison_models21:

    differences = (
        auc_seed_table21[model_name]
        - auc_seed_table21["Full_fusion"]
    )

    auc_difference_records21.append({
        "Model": model_name,
        "Mean_AUC_Difference_vs_Full": differences.mean(),
        "SD_AUC_Difference": differences.std(),
        "Seeds_Improved": int((differences > 0).sum()),
        "Seeds_Worse": int((differences < 0).sum()),
        "Seeds_Tied": int((differences == 0).sum())
    })

auc_difference_df21 = pd.DataFrame(
    auc_difference_records21
)

print("\nAUC DIFFERENCE COMPARED WITH FULL FUSION")
display(
    auc_difference_df21.round(4)
)

# ------------------------------------------------------------
# 13. COUNT HOW OFTEN EACH ABLATION BEATS FULL FUSION
# ------------------------------------------------------------

print("\nABLATION STABILITY SUMMARY")

for item in ABLATION_ITEMS:

    model_name = f"Fusion_without_{item}"

    differences = (
        auc_seed_table21[model_name]
        - auc_seed_table21["Full_fusion"]
    )

    wins = int((differences > 0).sum())
    losses = int((differences < 0).sum())
    ties = int((differences == 0).sum())

    print(
        f"{item}: "
        f"Improved={wins}/{len(RANDOM_SEEDS)}, "
        f"Worse={losses}/{len(RANDOM_SEEDS)}, "
        f"Tied={ties}/{len(RANDOM_SEEDS)}, "
        f"Mean AUC difference={differences.mean():.4f}"
    )

# ------------------------------------------------------------
# 14. SAVE RESULTS
# ------------------------------------------------------------

output_dir21 = RESULTS_ROOT / "ecog_ablation_stability"

output_dir21.mkdir(
    parents=True,
    exist_ok=True
)

experiment21_assessment_df.to_csv(
    output_dir21 / "ecog_ablation_stability_assessment_seed_results.csv",
    index=False
)

experiment21_participant_df.to_csv(
    output_dir21 / "ecog_ablation_stability_participant_seed_results.csv",
    index=False
)

participant_stability21.to_csv(
    output_dir21 / "ecog_ablation_stability_participant_stability.csv",
    index=False
)

assessment_stability21.to_csv(
    output_dir21 / "ecog_ablation_stability_assessment_stability.csv",
    index=False
)

auc_difference_df21.to_csv(
    output_dir21 / "ecog_ablation_stability_auc_differences.csv",
    index=False
)

auc_seed_table21.to_csv(
    output_dir21 / "ecog_ablation_stability_auc_by_seed.csv"
)

# Save all participant-level OOF predictions by seed/model.
oof_records21 = []

for (seed, model_name), probabilities in experiment21_oof.items():

    prediction_df = pd.DataFrame({
        ID_COL: groups21,
        TARGET_COL: y21,
        "Probability": probabilities
    })

    participant_prediction_df = (
        prediction_df
        .groupby(ID_COL)
        .agg({
            TARGET_COL: "first",
            "Probability": "mean"
        })
        .reset_index()
    )

    participant_prediction_df["Seed"] = seed
    participant_prediction_df["Model"] = model_name

    oof_records21.append(participant_prediction_df)

experiment21_oof_df = pd.concat(
    oof_records21,
    ignore_index=True
)

experiment21_oof_df.to_csv(
    output_dir21 / "ecog_ablation_stability_participant_oof_predictions.csv",
    index=False
)

print("\n" + "=" * 55)
print("STUDY 5: REPEATED-SEED ABLATION STABILITY COMPLETED")
print("=" * 55)
print("Results saved to:", output_dir21)



# ============================================================


Study 5: Repeated-Seed Ablation Stability cohort verified.
Assessments: 77
Participants: 71
CN assessments: 53
MCI assessments: 24

MODELS
Storyteller_only (77, 51)
Full_fusion (77, 63)
Fusion_without_ECOGPT1 (77, 62)
Fusion_without_ECOGPT2 (77, 62)
Fusion_without_ECOGPT8 (77, 62)
Fusion_without_ECOGPT9 (77, 62)

PARTICIPANT CLASS DISTRIBUTION
CN     49
MCI    22
Name: count, dtype: int64

RANDOM SEED: 11
Storyteller_only: AUC=0.6865, Balanced Acc=0.6150, MCI Recall=0.4545
Full_fusion: AUC=0.8126, Balanced Acc=0.7593, MCI Recall=0.6818
Fusion_without_ECOGPT1: AUC=0.8404, Balanced Acc=0.7899, MCI Recall=0.6818
Fusion_without_ECOGPT2: AUC=0.8256, Balanced Acc=0.7593, MCI Recall=0.6818
Fusion_without_ECOGPT8: AUC=0.7579, Balanced Acc=0.7263, MCI Recall=0.6364
Fusion_without_ECOGPT9: AUC=0.8163, Balanced Acc=0.7138, MCI Recall=0.5909

RANDOM SEED: 21
Storyteller_only: AUC=0.6772, Balanced Acc=0.6118, MCI Recall=0.5909
Full_fusion: AUC=0.8284, Balanced Acc=0.7491, MCI Recall=0.6818
Fusion_w

,Model,Accuracy_mean,Accuracy_std,Balanced_Accuracy_mean,Balanced_Accuracy_std,MCI_Precision_mean,MCI_Precision_std,MCI_Recall_mean,MCI_Recall_std,MCI_F1_mean,MCI_F1_std,Macro_F1_mean,Macro_F1_std,ROC_AUC_mean,ROC_AUC_std
0,Full_fusion,0.7634,0.0380,0.7271,0.0456,0.6162,0.0645,0.6318,0.0756,0.6227,0.0642,0.7250,0.0451,0.8222,0.0309
1,Fusion_without_ECOGPT1,0.7887,0.0435,0.7568,0.0445,0.6611,0.0864,0.6727,0.0636,0.6647,0.0625,0.7551,0.0474,0.8407,0.0259
2,Fusion_without_ECOGPT2,0.7803,0.0333,0.7431,0.0382,0.6471,0.0593,0.6455,0.0598,0.6454,0.0533,0.7430,0.0384,0.8406,0.0339
3,Fusion_without_ECOGPT8,0.7268,0.0283,0.6893,0.0373,0.5549,0.0418,0.5909,0.0643,0.5719,0.0505,0.6856,0.0347,0.7717,0.0322
4,Fusion_without_ECOGPT9,0.7493,0.0500,0.7232,0.0539,0.5883,0.0770,0.6545,0.0717,0.6188,0.0707,0.7159,0.0544,0.7919,0.0363
5,Storyteller_only,0.6380,0.0474,0.5962,0.0403,0.4321,0.0580,0.4864,0.0569,0.4552,0.0450,0.5915,0.0415,0.6792,0.0345



ASSESSMENT-LEVEL STABILITY RESULTS


,Model,Accuracy_mean,Accuracy_std,Balanced_Accuracy_mean,Balanced_Accuracy_std,MCI_Precision_mean,MCI_Precision_std,MCI_Recall_mean,MCI_Recall_std,MCI_F1_mean,MCI_F1_std,Macro_F1_mean,Macro_F1_std,ROC_AUC_mean,ROC_AUC_std
0,Full_fusion,0.7532,0.0401,0.7056,0.0458,0.6116,0.0728,0.5792,0.0693,0.5938,0.0659,0.7082,0.0469,0.8109,0.0309
1,Fusion_without_ECOGPT1,0.7779,0.0443,0.7349,0.0456,0.6575,0.0923,0.6208,0.0635,0.6364,0.0649,0.7381,0.0487,0.8307,0.0283
2,Fusion_without_ECOGPT2,0.7688,0.0351,0.7204,0.0385,0.6419,0.0669,0.5917,0.0549,0.6150,0.0553,0.7248,0.0402,0.8292,0.0340
3,Fusion_without_ECOGPT8,0.7195,0.0282,0.6708,0.0351,0.5507,0.0453,0.5417,0.0589,0.5456,0.0493,0.6713,0.0341,0.7561,0.0313
4,Fusion_without_ECOGPT9,0.7377,0.0485,0.7000,0.0513,0.5794,0.0794,0.6000,0.0657,0.5886,0.0690,0.6979,0.0528,0.7767,0.0363
5,Storyteller_only,0.6455,0.0494,0.6011,0.0416,0.4447,0.0642,0.4833,0.0527,0.4606,0.0462,0.5977,0.0433,0.6679,0.0301



AUC DIFFERENCE COMPARED WITH FULL FUSION


,Model,Mean_AUC_Difference_vs_Full,SD_AUC_Difference,Seeds_Improved,Seeds_Worse,Seeds_Tied
0,Storyteller_only,-0.1429,0.0195,0,10,0
1,Fusion_without_ECOGPT1,0.0186,0.0092,10,0,0
2,Fusion_without_ECOGPT2,0.0185,0.0142,9,1,0
3,Fusion_without_ECOGPT8,-0.0505,0.0143,0,10,0
4,Fusion_without_ECOGPT9,-0.0302,0.0140,1,9,0



ABLATION STABILITY SUMMARY
ECOGPT1: Improved=10/10, Worse=0/10, Tied=0/10, Mean AUC difference=0.0186
ECOGPT2: Improved=9/10, Worse=1/10, Tied=0/10, Mean AUC difference=0.0185
ECOGPT8: Improved=0/10, Worse=10/10, Tied=0/10, Mean AUC difference=-0.0505
ECOGPT9: Improved=1/10, Worse=9/10, Tied=0/10, Mean AUC difference=-0.0302

STUDY 5: REPEATED-SEED ABLATION STABILITY COMPLETED
Results saved to: c:\Users\amnar\Documents\ADNI4_Storyteller_ECOG_Classification\results\ecog_ablation_stability


## Study 6 — Nested Participant-Grouped Feature Selection

Runs 5-fold outer × 4-fold inner participant-grouped nested cross-validation across 10 seeds to obtain a leakage-free internal performance estimate for ECog-12 item-subset selection.


In [55]:
# Nested Participant Grouped Feature Selection NESTED PARTICIPANT-GROUPED FEATURE SELECTION
# ============================================================
#
# Objective:
# Test whether ECOG feature removal improves performance when
# feature selection is performed exclusively within training data.
#
# Cohort:
# Same fixed 30-day cohort as Studies 3-5.
# 77 assessments / 71 participants.
#
# Candidate feature sets:
# 1. Full fusion: all 12 ECOG items
# 2. Without ECOGPT1
# 3. Without ECOGPT2
# 4. Without ECOGPT1 and ECOGPT2
#
# Nested validation:
# Outer CV: 5-fold StratifiedGroupKFold
# Inner CV: 4-fold StratifiedGroupKFold
# Repeated with 10 random seeds.
#
# Feature selection criterion:
# Inner participant-level pooled OOF ROC-AUC.
#
# Primary evaluation:
# Outer participant-level OOF metrics.
#
# Important:
# - Participants never appear in both train and test folds.
# - Imputation and scaling happen inside each training fold.
# - Outer test data are not used for feature selection.
# - Repeated seeds measure split sensitivity, not independent
#   external validation.
# ============================================================

import numpy as np
import pandas as pd
import warnings

from pathlib import Path

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

OUTER_SPLITS = 5
INNER_SPLITS = 4

RANDOM_SEEDS22 = [
    11, 21, 42, 52, 62,
    72, 82, 92, 102, 112
]

# ------------------------------------------------------------
# 2. VERIFY AND REUSE THE STUDY 5: REPEATED-SEED ABLATION STABILITY COHORT
# ------------------------------------------------------------

required_variables22 = [
    "common_story",
    "window_ecog_data",
    "story_feature_cols",
    "ecog_feature_cols",
    "ID_COL",
    "TARGET_COL"
]

missing_variables22 = [
    name for name in required_variables22
    if name not in globals()
]

if missing_variables22:
    raise ValueError(
        "Missing required variables: "
        + str(missing_variables22)
    )

story22 = (
    common_story.copy()
    .reset_index(drop=True)
)

ecog22 = (
    window_ecog_data["30_days"]
    .copy()
    .reset_index(drop=True)
)

if len(story22) != len(ecog22):
    raise ValueError(
        "Storyteller and ECOG row counts do not match."
    )

if "ROW_ID" in ecog22.columns:
    if ecog22["ROW_ID"].tolist() != list(range(len(story22))):
        raise ValueError(
            "ECOG ROW_ID alignment does not match the cohort."
        )

if story22[ID_COL].isna().any():
    raise ValueError("Missing participant IDs detected.")

if story22[TARGET_COL].isna().any():
    raise ValueError("Missing target labels detected.")

# Check one consistent diagnosis label per participant.
participant_label_counts22 = (
    story22.groupby(ID_COL)[TARGET_COL].nunique()
)

if (participant_label_counts22 > 1).any():
    raise ValueError(
        "Participants with inconsistent diagnosis labels detected."
    )

# ------------------------------------------------------------
# 3. BUILD FEATURE MATRICES
# ------------------------------------------------------------

X_story22 = (
    story22[story_feature_cols]
    .apply(pd.to_numeric, errors="coerce")
    .reset_index(drop=True)
)

X_ecog22 = (
    ecog22[ecog_feature_cols]
    .apply(pd.to_numeric, errors="coerce")
    .reset_index(drop=True)
)

y22 = (
    story22[TARGET_COL]
    .astype(int)
    .to_numpy()
)

groups22 = story22[ID_COL].to_numpy()

if set(np.unique(y22)) != {0, 1}:
    raise ValueError(
        "Both CN=0 and MCI=1 must be present."
    )

print("STUDY 6: NESTED PARTICIPANT-GROUPED FEATURE SELECTION COHORT")
print("Assessments:", len(story22))
print("Participants:", pd.Series(groups22).nunique())
print("CN assessments:", int((y22 == 0).sum()))
print("MCI assessments:", int((y22 == 1).sum()))
print(
    "CN participants:",
    int(
        story22.loc[
            story22[TARGET_COL] == 0, ID_COL
        ].nunique()
    )
)
print(
    "MCI participants:",
    int(
        story22.loc[
            story22[TARGET_COL] == 1, ID_COL
        ].nunique()
    )
)

# ------------------------------------------------------------
# 4. DEFINE THE FOUR CANDIDATE FEATURE SETS
# ------------------------------------------------------------

X_full22 = pd.concat(
    [X_story22, X_ecog22],
    axis=1
)

ecog_without_1 = [
    col for col in ecog_feature_cols
    if col != "ECOGPT1"
]

ecog_without_2 = [
    col for col in ecog_feature_cols
    if col != "ECOGPT2"
]

ecog_without_1_2 = [
    col for col in ecog_feature_cols
    if col not in ["ECOGPT1", "ECOGPT2"]
]

candidate_features22 = {
    "Full_fusion": (
        list(story_feature_cols)
        + list(ecog_feature_cols)
    ),

    "Without_ECOGPT1": (
        list(story_feature_cols)
        + ecog_without_1
    ),

    "Without_ECOGPT2": (
        list(story_feature_cols)
        + ecog_without_2
    ),

    "Without_ECOGPT1_ECOGPT2": (
        list(story_feature_cols)
        + ecog_without_1_2
    )
}

X_candidates22 = {
    name: pd.concat(
        [
            X_story22,
            X_ecog22[
                [
                    col for col in feature_cols
                    if col in ecog_feature_cols
                ]
            ]
        ],
        axis=1
    ) if name == "Full_fusion" else
    pd.concat(
        [
            X_story22,
            X_ecog22[
                [
                    col for col in feature_cols
                    if col in ecog_feature_cols
                ]
            ]
        ],
        axis=1
    )
    for name, feature_cols in candidate_features22.items()
}

# Ensure exact intended feature columns and order.
for name, feature_cols in candidate_features22.items():
    X_candidates22[name] = (
        X_story22
        .join(X_ecog22[[
            col for col in feature_cols
            if col in ecog_feature_cols
        ]])
        .loc[:, feature_cols]
    )

print("\nCANDIDATE FEATURE SETS")

for name, X in X_candidates22.items():
    print(f"{name}: {X.shape[1]} features")

# ------------------------------------------------------------
# 5. MODEL PIPELINE
# ------------------------------------------------------------

def make_experiment22_model():

    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
                random_state=42
            )
        )
    ])

# ------------------------------------------------------------
# 6. METRIC FUNCTION
# ------------------------------------------------------------

def metrics_experiment22(y_true, probabilities):

    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities)

    predictions = (
        probabilities >= 0.50
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "Accuracy": accuracy_score(
            y_true, predictions
        ),

        "Balanced_Accuracy": balanced_accuracy_score(
            y_true, predictions
        ),

        "MCI_Precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "MCI_Recall": recall_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "MCI_F1": f1_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "Macro_F1": f1_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0
        ),

        "ROC_AUC": (
            roc_auc_score(y_true, probabilities)
            if len(np.unique(y_true)) == 2
            else np.nan
        ),

        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

# ------------------------------------------------------------
# 7. PARTICIPANT-LEVEL AGGREGATION
# ------------------------------------------------------------

def aggregate_participant_predictions22(
    participant_ids,
    true_labels,
    probabilities
):

    df = pd.DataFrame({
        "PTID": participant_ids,
        "TARGET": true_labels,
        "Probability": probabilities
    })

    # Every participant must have one consistent label.
    label_counts = df.groupby("PTID")["TARGET"].nunique()

    if (label_counts > 1).any():
        raise ValueError(
            "Inconsistent participant labels during aggregation."
        )

    result = (
        df.groupby("PTID")
        .agg({
            "TARGET": "first",
            "Probability": "mean"
        })
        .reset_index()
    )

    return result

# ------------------------------------------------------------
# 8. NESTED CROSS-VALIDATION
# ------------------------------------------------------------

outer_records22 = []
inner_selection_records22 = []
fold_selection_records22 = []

experiment22_oof = {}

for seed in RANDOM_SEEDS22:

    print("\n" + "=" * 65)
    print("OUTER RANDOM SEED:", seed)
    print("=" * 65)

    outer_cv = StratifiedGroupKFold(
        n_splits=OUTER_SPLITS,
        shuffle=True,
        random_state=seed
    )

    outer_folds = list(
        outer_cv.split(
            X_story22,
            y22,
            groups22
        )
    )

    # Store outer OOF predictions for each fixed candidate.
    outer_predictions = {
        name: np.full(len(y22), np.nan)
        for name in X_candidates22
    }

    # Store predictions from the nested selected model.
    nested_selected_predictions = np.full(
        len(y22),
        np.nan
    )

    # --------------------------------------------------------
    # OUTER FOLD LOOP
    # --------------------------------------------------------

    for outer_fold, (outer_train_idx, outer_test_idx) in enumerate(
        outer_folds, 1
    ):

        print(
            f"\nOuter fold {outer_fold}/{OUTER_SPLITS}"
        )

        outer_train_ids = set(
            groups22[outer_train_idx]
        )

        outer_test_ids = set(
            groups22[outer_test_idx]
        )

        # Outer participant leakage check.
        assert outer_train_ids.isdisjoint(
            outer_test_ids
        )

        X_train_base = X_story22.iloc[
            outer_train_idx
        ]

        y_train = y22[outer_train_idx]

        groups_train = groups22[outer_train_idx]

        # ----------------------------------------------------
        # INNER CROSS-VALIDATION
        # ----------------------------------------------------

        inner_cv = StratifiedGroupKFold(
            n_splits=INNER_SPLITS,
            shuffle=True,
            random_state=seed + outer_fold
        )

        inner_folds = list(
            inner_cv.split(
                X_train_base,
                y_train,
                groups_train
            )
        )

        # Confirm inner participant separation.
        for inner_train_idx, inner_val_idx in inner_folds:

            inner_train_ids = set(
                groups_train[inner_train_idx]
            )

            inner_val_ids = set(
                groups_train[inner_val_idx]
            )

            assert inner_train_ids.isdisjoint(
                inner_val_ids
            )

        inner_candidate_scores = {}

        # Evaluate each candidate on identical inner folds.
        for candidate_name, X_candidate in X_candidates22.items():

            X_outer_train = X_candidate.iloc[
                outer_train_idx
            ]

            inner_oof_probabilities = np.full(
                len(outer_train_idx),
                np.nan
            )

            for inner_train_idx, inner_val_idx in inner_folds:

                model = make_experiment22_model()

                model.fit(
                    X_outer_train.iloc[inner_train_idx],
                    y_train[inner_train_idx]
                )

                probabilities = model.predict_proba(
                    X_outer_train.iloc[inner_val_idx]
                )[:, 1]

                inner_oof_probabilities[inner_val_idx] = (
                    probabilities
                )

            if np.isnan(inner_oof_probabilities).any():
                raise ValueError(
                    "Missing inner OOF predictions."
                )

            # Participant-level inner OOF evaluation.
            inner_participant_df = (
                aggregate_participant_predictions22(
                    groups_train,
                    y_train,
                    inner_oof_probabilities
                )
            )

            inner_auc = roc_auc_score(
                inner_participant_df["TARGET"],
                inner_participant_df["Probability"]
            )

            inner_candidate_scores[candidate_name] = inner_auc

            inner_selection_records22.append({
                "Seed": seed,
                "Outer_Fold": outer_fold,
                "Candidate": candidate_name,
                "Inner_Participant_AUC": inner_auc
            })

        # ----------------------------------------------------
        # SELECT BEST FEATURE SET USING INNER DATA ONLY
        # ----------------------------------------------------

        selected_candidate = max(
            inner_candidate_scores,
            key=inner_candidate_scores.get
        )

        selected_inner_auc = (
            inner_candidate_scores[selected_candidate]
        )

        print("Inner candidate AUC scores:")

        for candidate_name, score in inner_candidate_scores.items():
            print(
                f"  {candidate_name}: {score:.4f}"
            )

        print(
            "Selected candidate:",
            selected_candidate
        )

        fold_selection_records22.append({
            "Seed": seed,
            "Outer_Fold": outer_fold,
            "Selected_Candidate": selected_candidate,
            "Selected_Inner_AUC": selected_inner_auc
        })

        # ----------------------------------------------------
        # OUTER TEST EVALUATION
        # ----------------------------------------------------
        #
        # Fit each candidate on the full outer training set.
        # The outer test set is not used to select features.
        #
        # The nested selected prediction is taken from the
        # candidate selected using inner validation only.
        # ----------------------------------------------------

        for candidate_name, X_candidate in X_candidates22.items():

            X_train = X_candidate.iloc[
                outer_train_idx
            ]

            X_test = X_candidate.iloc[
                outer_test_idx
            ]

            outer_model = make_experiment22_model()

            outer_model.fit(
                X_train,
                y_train
            )

            test_probabilities = (
                outer_model.predict_proba(X_test)[:, 1]
            )

            outer_predictions[candidate_name][
                outer_test_idx
            ] = test_probabilities

            if candidate_name == selected_candidate:

                nested_selected_predictions[
                    outer_test_idx
                ] = test_probabilities

    # --------------------------------------------------------
    # 9. SUMMARIZE OUTER OOF RESULTS FOR THIS SEED
    # --------------------------------------------------------

    if np.isnan(nested_selected_predictions).any():
        raise ValueError(
            "Missing nested selected outer OOF predictions."
        )

    # Fixed candidate results on outer test folds.
    for candidate_name, probabilities in outer_predictions.items():

        if np.isnan(probabilities).any():
            raise ValueError(
                f"Missing outer predictions: {candidate_name}"
            )

        participant_df = (
            aggregate_participant_predictions22(
                groups22,
                y22,
                probabilities
            )
        )

        participant_metrics = metrics_experiment22(
            participant_df["TARGET"],
            participant_df["Probability"]
        )

        outer_records22.append({
            "Seed": seed,
            "Model": candidate_name,
            "Evaluation": "Participant-level",
            "N": len(participant_df),
            **participant_metrics
        })

        experiment22_oof[
            (seed, candidate_name)
        ] = probabilities.copy()

    # Nested selection strategy.
    nested_participant_df = (
        aggregate_participant_predictions22(
            groups22,
            y22,
            nested_selected_predictions
        )
    )

    nested_metrics = metrics_experiment22(
        nested_participant_df["TARGET"],
        nested_participant_df["Probability"]
    )

    outer_records22.append({
        "Seed": seed,
        "Model": "Nested_selected",
        "Evaluation": "Participant-level",
        "N": len(nested_participant_df),
        **nested_metrics
    })

    experiment22_oof[
        (seed, "Nested_selected")
    ] = nested_selected_predictions.copy()

    # Secondary assessment-level results for nested strategy.
    assessment_metrics = metrics_experiment22(
        y22,
        nested_selected_predictions
    )

    outer_records22.append({
        "Seed": seed,
        "Model": "Nested_selected",
        "Evaluation": "Assessment-level",
        "N": len(y22),
        **assessment_metrics
    })

    print(
        f"\nSeed {seed} nested participant AUC: "
        f"{nested_metrics['ROC_AUC']:.4f}"
    )

    print(
        f"Seed {seed} nested balanced accuracy: "
        f"{nested_metrics['Balanced_Accuracy']:.4f}"
    )

    print(
        f"Seed {seed} nested MCI recall: "
        f"{nested_metrics['MCI_Recall']:.4f}"
    )

# ------------------------------------------------------------
# 10. CREATE RESULTS DATAFRAMES
# ------------------------------------------------------------

experiment22_results_df = pd.DataFrame(
    outer_records22
)

experiment22_inner_df = pd.DataFrame(
    inner_selection_records22
)

experiment22_selection_df = pd.DataFrame(
    fold_selection_records22
)

print("\nNESTED CV COMPLETED")
print("Random seeds:", len(RANDOM_SEEDS22))
print("Outer folds per seed:", OUTER_SPLITS)
print("Inner folds per outer fold:", INNER_SPLITS)

# ------------------------------------------------------------
# 11. PARTICIPANT-LEVEL OUTER PERFORMANCE
# ------------------------------------------------------------

participant_results22 = (
    experiment22_results_df[
        experiment22_results_df["Evaluation"]
        == "Participant-level"
    ]
    .copy()
)

metric_columns22 = [
    "Accuracy",
    "Balanced_Accuracy",
    "MCI_Precision",
    "MCI_Recall",
    "MCI_F1",
    "Macro_F1",
    "ROC_AUC"
]

participant_summary22 = (
    participant_results22
    .groupby("Model")[metric_columns22]
    .agg(["mean", "std"])
)

participant_summary22.columns = [
    f"{metric}_{stat}"
    for metric, stat in participant_summary22.columns
]

participant_summary22 = (
    participant_summary22.reset_index()
)

print("\nPARTICIPANT-LEVEL OUTER PERFORMANCE")
display(
    participant_summary22.round(4)
)

# ------------------------------------------------------------
# 12. ASSESSMENT-LEVEL NESTED PERFORMANCE
# ------------------------------------------------------------

assessment_results22 = (
    experiment22_results_df[
        experiment22_results_df["Evaluation"]
        == "Assessment-level"
    ]
    .copy()
)

print("\nASSESSMENT-LEVEL NESTED PERFORMANCE")

display(
    assessment_results22[
        [
            "Seed",
            "Accuracy",
            "Balanced_Accuracy",
            "MCI_Precision",
            "MCI_Recall",
            "MCI_F1",
            "ROC_AUC"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 13. FEATURE-SELECTION FREQUENCY
# ------------------------------------------------------------

selection_frequency22 = (
    experiment22_selection_df[
        "Selected_Candidate"
    ]
    .value_counts()
    .rename_axis("Selected_Candidate")
    .reset_index(name="Selection_Count")
)

selection_frequency22["Selection_Percentage"] = (
    selection_frequency22["Selection_Count"]
    / len(experiment22_selection_df)
    * 100
)

print("\nFEATURE SET SELECTION FREQUENCY")
display(
    selection_frequency22.round(2)
)

# ------------------------------------------------------------
# 14. INNER VALIDATION SCORES
# ------------------------------------------------------------

inner_score_summary22 = (
    experiment22_inner_df
    .groupby("Candidate")["Inner_Participant_AUC"]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
)

print("\nINNER VALIDATION AUC SUMMARY")
display(
    inner_score_summary22.round(4)
)

# ------------------------------------------------------------
# 15. COMPARE NESTED SELECTION WITH FULL FUSION
# ------------------------------------------------------------

participant_auc_by_seed22 = (
    participant_results22
    .pivot(
        index="Seed",
        columns="Model",
        values="ROC_AUC"
    )
)

if (
    "Nested_selected" in participant_auc_by_seed22.columns
    and "Full_fusion" in participant_auc_by_seed22.columns
):

    nested_minus_full22 = (
        participant_auc_by_seed22["Nested_selected"]
        - participant_auc_by_seed22["Full_fusion"]
    )

    nested_comparison22 = pd.DataFrame([{
        "Mean_AUC_Difference": nested_minus_full22.mean(),
        "SD_AUC_Difference": nested_minus_full22.std(),
        "Seeds_Nested_Better": int(
            (nested_minus_full22 > 0).sum()
        ),
        "Seeds_Nested_Worse": int(
            (nested_minus_full22 < 0).sum()
        ),
        "Seeds_Tied": int(
            (nested_minus_full22 == 0).sum()
        )
    }])

    print("\nNESTED SELECTION VS FULL FUSION")
    display(
        nested_comparison22.round(4)
    )

# ------------------------------------------------------------
# 16. SAVE ALL RESULTS
# ------------------------------------------------------------

output_dir22 = RESULTS_ROOT / "nested_feature_selection"

output_dir22.mkdir(
    parents=True,
    exist_ok=True
)

experiment22_results_df.to_csv(
    output_dir22 / "nested_feature_selection_outer_results.csv",
    index=False
)

experiment22_inner_df.to_csv(
    output_dir22 / "nested_feature_selection_inner_validation_scores.csv",
    index=False
)

experiment22_selection_df.to_csv(
    output_dir22 / "nested_feature_selection_feature_selection_frequency.csv",
    index=False
)

participant_summary22.to_csv(
    output_dir22 / "nested_feature_selection_participant_summary.csv",
    index=False
)

inner_score_summary22.to_csv(
    output_dir22 / "nested_feature_selection_inner_auc_summary.csv",
    index=False
)

selection_frequency22.to_csv(
    output_dir22 / "nested_feature_selection_selection_frequency_summary.csv",
    index=False
)

if "nested_comparison22" in globals():
    nested_comparison22.to_csv(
        output_dir22 / "nested_feature_selection_nested_vs_full.csv",
        index=False
    )

# Save outer OOF probabilities.
oof_records22 = []

for (seed, model_name), probabilities in experiment22_oof.items():

    participant_df = (
        aggregate_participant_predictions22(
            groups22,
            y22,
            probabilities
        )
    )

    participant_df["Seed"] = seed
    participant_df["Model"] = model_name

    oof_records22.append(participant_df)

experiment22_oof_df = pd.concat(
    oof_records22,
    ignore_index=True
)

experiment22_oof_df.to_csv(
    output_dir22 / "nested_feature_selection_participant_oof_predictions.csv",
    index=False
)

print("\n" + "=" * 65)
print("STUDY 6: NESTED PARTICIPANT-GROUPED FEATURE SELECTION COMPLETED")
print("=" * 65)
print("Results saved to:", output_dir22)


# Nested Participant Grouped Feature Selection Nested Participant-Grouped Feature Selection
# 
# Study 6: Nested Participant-Grouped Feature Selection evaluated whether ECOG feature reduction could improve CN–MCI classification when feature selection was conducted exclusively within training data. A nested participant-grouped cross-validation strategy was implemented using five outer folds and four inner folds, repeated across ten random seeds. Four feature configurations were compared: full fusion, fusion without ECOGPT1, fusion without ECOGPT2, and fusion without both items.
# 
# The full fusion model achieved a mean participant-level ROC-AUC of 0.8222 (SD = 0.0309). The nested feature-selection strategy achieved a higher mean ROC-AUC of 0.8412 (SD = 0.0300), with a mean balanced accuracy of 0.7543 and MCI recall of 0.6636. The nested strategy produced a higher AUC than full fusion across all ten random seeds.
# 
# The reduced model excluding both ECOGPT1 and ECOGPT2 achieved the highest mean ROC-AUC among the fixed candidate configurations at 0.8583 (SD = 0.0276). Analysis of feature-selection frequency showed that the configuration excluding both items was selected in 48% of outer folds, while the configuration excluding ECOGPT1 alone was selected in 30%. ECOGPT1 was excluded in 78% of selections, compared with 60% for ECOGPT2.
# 
# These findings suggest that ECOGPT1, and potentially ECOGPT2, may contain redundant or less useful predictive information in the evaluated cohort. Nevertheless, the results are based on repeated internal validation of the same 71 participants. Furthermore, the candidate feature configurations were motivated by earlier experiments on this cohort. Therefore, the findings should be considered exploratory and require independent validation before the reduced feature configuration can be considered generally superior.



# ============================================================


STUDY 6: NESTED PARTICIPANT-GROUPED FEATURE SELECTION COHORT
Assessments: 77
Participants: 71
CN assessments: 53
MCI assessments: 24
CN participants: 49
MCI participants: 22

CANDIDATE FEATURE SETS
Full_fusion: 63 features
Without_ECOGPT1: 62 features
Without_ECOGPT2: 62 features
Without_ECOGPT1_ECOGPT2: 61 features

OUTER RANDOM SEED: 11

Outer fold 1/5
Inner candidate AUC scores:
  Full_fusion: 0.7949
  Without_ECOGPT1: 0.7964
  Without_ECOGPT2: 0.8296
  Without_ECOGPT1_ECOGPT2: 0.8296
Selected candidate: Without_ECOGPT2

Outer fold 2/5
Inner candidate AUC scores:
  Full_fusion: 0.8319
  Without_ECOGPT1: 0.8431
  Without_ECOGPT2: 0.8250
  Without_ECOGPT1_ECOGPT2: 0.8389
Selected candidate: Without_ECOGPT1

Outer fold 3/5
Inner candidate AUC scores:
  Full_fusion: 0.7691
  Without_ECOGPT1: 0.7838
  Without_ECOGPT2: 0.7676
  Without_ECOGPT1_ECOGPT2: 0.7882
Selected candidate: Without_ECOGPT1_ECOGPT2

Outer fold 4/5
Inner candidate AUC scores:
  Full_fusion: 0.9167
  Without_ECOGPT1: 0.

,Model,Accuracy_mean,Accuracy_std,Balanced_Accuracy_mean,Balanced_Accuracy_std,MCI_Precision_mean,MCI_Precision_std,MCI_Recall_mean,MCI_Recall_std,MCI_F1_mean,MCI_F1_std,Macro_F1_mean,Macro_F1_std,ROC_AUC_mean,ROC_AUC_std
0,Full_fusion,0.7634,0.0380,0.7271,0.0456,0.6162,0.0645,0.6318,0.0756,0.6227,0.0642,0.7250,0.0451,0.8222,0.0309
1,Nested_selected,0.7887,0.0409,0.7543,0.0457,0.6607,0.0741,0.6636,0.0684,0.6608,0.0633,0.7536,0.0464,0.8412,0.0300
2,Without_ECOGPT1,0.7887,0.0435,0.7568,0.0445,0.6611,0.0864,0.6727,0.0636,0.6647,0.0625,0.7551,0.0474,0.8407,0.0259
3,Without_ECOGPT1_ECOGPT2,0.7944,0.0306,0.7646,0.0354,0.6648,0.0578,0.6864,0.0623,0.6739,0.0487,0.7618,0.0350,0.8583,0.0276
4,Without_ECOGPT2,0.7803,0.0333,0.7431,0.0382,0.6471,0.0593,0.6455,0.0598,0.6454,0.0533,0.7430,0.0384,0.8406,0.0339



ASSESSMENT-LEVEL NESTED PERFORMANCE


,Seed,Accuracy,Balanced_Accuracy,MCI_Precision,MCI_Recall,MCI_F1,ROC_AUC
5,11,0.7792,0.7370,0.6522,0.6250,0.6383,0.8113
11,21,0.7662,0.7276,0.6250,0.6250,0.6250,0.8388
17,42,0.7792,0.7484,0.6400,0.6667,0.6531,0.8325
23,52,0.8052,0.7673,0.6957,0.6667,0.6809,0.8105
29,62,0.7662,0.7048,0.6500,0.5417,0.5909,0.7987
35,72,0.6883,0.6368,0.5000,0.5000,0.5000,0.7932
41,82,0.8442,0.7956,0.8000,0.6667,0.7273,0.8491
47,92,0.7403,0.6859,0.5909,0.5417,0.5652,0.8011
53,102,0.7922,0.7579,0.6667,0.6667,0.6667,0.8719
59,112,0.8052,0.7445,0.7368,0.5833,0.6512,0.8899



FEATURE SET SELECTION FREQUENCY


,Selected_Candidate,Selection_Count,Selection_Percentage
0,Without_ECOGPT1_ECOGPT2,24,48.0
1,Without_ECOGPT1,15,30.0
2,Without_ECOGPT2,6,12.0
3,Full_fusion,5,10.0



INNER VALIDATION AUC SUMMARY


,Candidate,mean,std,min,max
0,Full_fusion,0.7870,0.0571,0.6606,0.9403
1,Without_ECOGPT1,0.7973,0.0566,0.6637,0.9375
2,Without_ECOGPT1_ECOGPT2,0.8020,0.0511,0.6878,0.9236
3,Without_ECOGPT2,0.7910,0.0553,0.6735,0.9486



NESTED SELECTION VS FULL FUSION


,Mean_AUC_Difference,SD_AUC_Difference,Seeds_Nested_Better,Seeds_Nested_Worse,Seeds_Tied
0,0.019,0.0109,10,0,0



STUDY 6: NESTED PARTICIPANT-GROUPED FEATURE SELECTION COMPLETED
Results saved to: c:\Users\amnar\Documents\ADNI4_Storyteller_ECOG_Classification\results\nested_feature_selection
